In [5]:
import requests
import pandas as pd
import zipfile
import io
from io import StringIO
import time
from datetime import datetime, timedelta
import json
import openpyxl

In [31]:
API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"

url = "https://api.pjm.com/api/v1/rt_hrl_lmps"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

def format_date(dt):
    """Format date as m-d-yyyy with no leading zeros."""
    return f"{dt.month}-{dt.day}-{dt.year}"

def find_latest_date_with_data(days_back=7):
    """Walk backwards from today until we find a date with data."""
    for i in range(1, days_back + 1):
        date = datetime.now() - timedelta(days=i)
        date_str = format_date(date)
        r = requests.get(url, headers=headers, params={
            "rowCount": 1,
            "startRow": 1,
            "datetime_beginning_ept": f"{date_str} 00:00 to {date_str} 23:00",
            "row_is_current": 1,
        })
        if r.status_code != 200 or not r.text.strip():
            print(f"{date_str} -> bad response: {r.status_code}")
            continue
        total = r.json().get("totalRows", 0)
        print(f"{date_str} -> totalRows: {total}")
        if total > 0:
            return date_str
    return None

def get_latest_lmps(n=10):
    date_str = find_latest_date_with_data()
    if not date_str:
        print("No data found in the last 7 days")
        return

    r = requests.get(url, headers=headers, params={
        "rowCount": n,
        "startRow": 1,
        "sort": "datetime_beginning_ept",
        "order": "Desc",
        "datetime_beginning_ept": f"{date_str} 00:00 to {date_str} 23:00",
        "row_is_current": 1,
    })
    
    if not r.text.strip():
        print("Empty response on final fetch")
        return

    data = r.json()
    items = data.get("items", [])
    print(f"\nLatest {n} rows from {date_str}:\n")
    for row in items:
        print(row)

get_latest_lmps(10)

3-21-2026 -> totalRows: 0
3-20-2026 -> totalRows: 0
3-19-2026 -> totalRows: 344952

Latest 10 rows from 3-19-2026:

{'datetime_beginning_utc': '2026-03-20T03:00:00', 'datetime_beginning_ept': '2026-03-19T23:00:00', 'pnode_id': 1, 'pnode_name': 'PJM-RTO', 'voltage': None, 'equipment': None, 'type': 'ZONE', 'zone': None, 'system_energy_price_rt': 28.04, 'total_lmp_rt': 28.122513, 'congestion_price_rt': 0.066568, 'marginal_loss_price_rt': 0.016779, 'row_is_current': True, 'version_nbr': 1}
{'datetime_beginning_utc': '2026-03-20T03:00:00', 'datetime_beginning_ept': '2026-03-19T23:00:00', 'pnode_id': 3, 'pnode_name': 'MID-ATL/APS', 'voltage': None, 'equipment': None, 'type': 'ZONE', 'zone': None, 'system_energy_price_rt': 28.04, 'total_lmp_rt': 26.272631, 'congestion_price_rt': -2.126578, 'marginal_loss_price_rt': 0.360043, 'row_is_current': True, 'version_nbr': 1}
{'datetime_beginning_utc': '2026-03-20T03:00:00', 'datetime_beginning_ept': '2026-03-19T23:00:00', 'pnode_id': 48592, 'pnode_na

In [35]:
API_KEY = "d14f4a5d0aa6498a84ede0d1d5401e10"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

# Check raw response first
r = requests.get(
    "https://api.pjm.com/api/v1/pnode",
    headers=headers,
    params={
        "rowCount": 50000,
        "startRow": 1,
        "zone": "DOM",
        "terminate_date_ept": "12/31/9999exact",
    }
)
print("Raw response:", r.text[:500])

# Try without the termination date filter — just get all DOM nodes
print("\n=== Without termination filter ===")
r2 = requests.get(
    "https://api.pjm.com/api/v1/pnode",
    headers=headers,
    params={
        "rowCount": 50000,
        "startRow": 1,
        "zone": "DOM",
    }
)
data = r2.json()
print("Total rows:", data.get("totalRows"))
df = pd.DataFrame(data.get("items", []))
print("Columns:", df.columns.tolist())
print(df.head(5))
df.to_csv("dom_pnodes.csv", index=False)
print("\nSaved to dom_pnodes.csv")

Raw response: {"feedMetadata":{"id":50,"feedId":"111","name":"pnode","displayName":"Pricing Nodes","description":"This data contains master information on pnodes. A pnode is a single pricing node or subset of pricing nodes where a physical injection or withdrawal is modeled and for which a Locational Marginal Price is calculated and used for financial settlements.","category":"Reference Data","firstAvailable":"1998-04-01T00:00:00","version":"1","internalName":"f_111_pnode","defaultSearchJson":"{\"rowCount\": 

=== Without termination filter ===
Total rows: 2318
Columns: ['pnode_id', 'pnode_name', 'pnode_type', 'pnode_subtype', 'zone', 'voltage_level', 'effective_date', 'termination_date']
   pnode_id                pnode_name pnode_type pnode_subtype zone  \
0  32418803      CAYUGA2 345 KV  CAY2        BUS           EXT  DOM   
1  34884747  TILLERY 115 KV  4_TOPOLO        BUS           EXT  DOM   
2  34885205  ALTAVSTA13 KV   TX2TLOAD        BUS          LOAD  DOM   
3  34885227       

In [36]:
df = pd.read_csv("dom_pnodes.csv")

# Filter for active nodes (termination date = 9999)
df_active = df[df["termination_date"].str.startswith("9999")]
print("Total DOM nodes:", len(df))
print("Active DOM nodes:", len(df_active))

# Look at subtype breakdown
print("\nSubtype breakdown:")
print(df_active["pnode_subtype"].value_counts())

print("\nSample active nodes:")
print(df_active[["pnode_id", "pnode_name", "pnode_subtype", "voltage_level"]].head(20))

Total DOM nodes: 2318
Active DOM nodes: 1755

Subtype breakdown:
pnode_subtype
LOAD    1336
GEN      380
EHV       38
EXT        1
Name: count, dtype: int64

Sample active nodes:
     pnode_id                pnode_name pnode_subtype voltage_level
563  34885183       ACCA    13 KV   TX1          LOAD         13 KV
564  34885185       ACCA    13 KV   TX2          LOAD         13 KV
565  34885187       ACCA    35 KV   TX3          LOAD         35 KV
566  34885189       ACCA    35 KV   TX4          LOAD         35 KV
567  34885191       ACCA    35 KV   TX5          LOAD         35 KV
568  34885193       ALLIED  13 KV   TX1          LOAD         13 KV
569  34885195       ALLIED  13 KV   TX2          LOAD         13 KV
570  34885197       ALLIED  13 KV   TX3          LOAD         13 KV
571  34885199       ALPINE  13 KV   TX1          LOAD         13 KV
572  34885201       ALPINE  13 KV   TX2          LOAD         13 KV
573  34885203       ALPINE  13 KV   TX3          LOAD         13 KV
574  

In [38]:
# Download EIA-860 generator data (has substation/plant locations with state)
# This is the most recent available year
url = "https://www.eia.gov/electricity/data/eia860/archive/xls/eia8602022.zip"

print("Downloading EIA-860 data...")
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

# List files in the zip to find the right one
print("\nFiles in zip:")
for name in z.namelist():
    print(name)


Files in zip:
1___Utility_Y2022.xlsx
2___Plant_Y2022.xlsx
3_1_Generator_Y2022.xlsx
3_2_Wind_Y2022.xlsx
3_3_Solar_Y2022.xlsx
3_4_Energy_Storage_Y2022.xlsx
3_5_Multifuel_Y2022.xlsx
4___Owner_Y2022.xlsx
6_1_EnviroAssoc_Y2022.xlsx
6_2_EnviroEquip_Y2022.xlsx
EIA-860 Form 2022.xlsx
EIA-860 Instructions.pdf
LayoutY2022.xlsx


In [6]:
# Load the plant file which has location data
url = "https://www.eia.gov/electricity/data/eia860/archive/xls/eia8602022.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

# Read plant file
with z.open("2___Plant_Y2022.xlsx") as f:
    df_plants = pd.read_excel(f, sheet_name="Plant", header=1)

print("Plant file columns:")
print(df_plants.columns.tolist())
print("\nShape:", df_plants.shape)
print("\nSample:")
print(df_plants.head(5))

# Filter for Virginia plants on Dominion's grid
df_va = df_plants[
    (df_plants["State"] == "VA") & 
    (df_plants["Balancing Authority Code"] == "PJM")
]
print(f"\nVirginia PJM plants: {len(df_va)}")
print(df_va[["Plant Name", "State", "Balancing Authority Code", 
             "Transmission or Distribution System Owner"]].head(20))

Plant file columns:
['Utility ID', 'Utility Name', 'Plant Code', 'Plant Name', 'Street Address', 'City', 'State', 'Zip', 'County', 'Latitude', 'Longitude', 'NERC Region', 'Balancing Authority Code', 'Balancing Authority Name', 'Name of Water Source', 'Primary Purpose (NAICS Code)', 'Regulatory Status', 'Sector', 'Sector Name', 'FERC Cogeneration Status', 'FERC Cogeneration Docket Number', 'FERC Small Power Producer Status', 'FERC Small Power Producer Docket Number', 'FERC Exempt Wholesale Generator Status', 'FERC Exempt Wholesale Generator Docket Number', 'Ash Impoundment?', 'Ash Impoundment Lined?', 'Ash Impoundment Status', 'Transmission or Distribution System Owner', 'Transmission or Distribution System Owner ID', 'Transmission or Distribution System Owner State', 'Grid Voltage (kV)', 'Grid Voltage 2 (kV)', 'Grid Voltage 3 (kV)', 'Energy Storage', 'Natural Gas LDC Name', 'Natural Gas Pipeline Name 1', 'Natural Gas Pipeline Name 2', 'Natural Gas Pipeline Name 3', 'Pipeline Notes', 'N

In [7]:
import subprocess
subprocess.run([
    "/Users/elenamurray/miniconda3/envs/mds_thesis/bin/pip", 
    "install", "geopy", "shapely", "geopandas"
])

  Using cached geopy-2.4.1-py3-none-any.whl.metadata (6.8 kB)
Using cached geopy-2.4.1-py3-none-any.whl (125 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 11.8 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 9.7 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [geopandas]/6 [geopy]o]


CompletedProcess(args=['/Users/elenamurray/miniconda3/envs/mds_thesis/bin/pip', 'install', 'geopy', 'shapely', 'geopandas'], returncode=0)

In [8]:
import pandas as pd
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

df_dom = pd.read_csv("dom_pnodes.csv")
df_active = df_dom[df_dom["termination_date"].str.startswith("9999")].copy()

# Clean pnode names for geocoding
df_active["name_clean"] = (df_active["pnode_name"]
    .str.replace(r'\d+\s*KV.*$', '', regex=True)
    .str.strip()
)

# Set up geocoder with rate limiting
geolocator = Nominatim(user_agent="pjm_node_locator")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Test on a handful of well-known Virginia substations first
test_nodes = [
    "ACCA",
    "BELLWOOD", 
    "POSSUM POINT",
    "CHESTERFIELD",
    "SMITH MOUNTAIN"
]

print("Testing geocoding on sample nodes:\n")
for name in test_nodes:
    query = f"{name} substation Virginia"
    location = geocode(query)
    if location:
        print(f"{name}: {location.latitude:.4f}, {location.longitude:.4f} -> {location.address}")
    else:
        print(f"{name}: not found")
    time.sleep(1)

Testing geocoding on sample nodes:

ACCA: not found
BELLWOOD: 37.4174, -77.4327 -> Bellwood Substation, 2506, Myron Avenue, Bellwood Terrace, Bellwood, Chesterfield County, Virginia, 23237, United States
POSSUM POINT: not found
CHESTERFIELD: not found
SMITH MOUNTAIN: not found


In [9]:
import pandas as pd
import zipfile
import io
import requests

# Load EIA plant file we already downloaded
url = "https://www.eia.gov/electricity/data/eia860/archive/xls/eia8602022.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

with z.open("2___Plant_Y2022.xlsx") as f:
    df_plants = pd.read_excel(f, sheet_name="Plant", header=1)

# Check DOM zone breakdown by state
df_dom_eia = df_plants[df_plants["Balancing Authority Code"] == "PJM"]
df_dom_eia = df_dom_eia[df_dom_eia["Transmission or Distribution System Owner"].str.contains(
    "Virginia Electric|Dominion", case=False, na=False
)]

print("State breakdown for Dominion/Virginia Electric plants in PJM:")
print(df_dom_eia["State"].value_counts())
print(f"\nTotal plants: {len(df_dom_eia)}")

State breakdown for Dominion/Virginia Electric plants in PJM:
State
VA    171
NC    113
WV      4
Name: count, dtype: int64

Total plants: 288


In [10]:
import pandas as pd
import geopandas as gpd
import zipfile
import io
import requests
from shapely.geometry import Point

# Load EIA plant file
url = "https://www.eia.gov/electricity/data/eia860/archive/xls/eia8602022.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

with z.open("2___Plant_Y2022.xlsx") as f:
    df_plants = pd.read_excel(f, sheet_name="Plant", header=1)

# Get Dominion plants with coordinates
df_dom_eia = df_plants[
    df_plants["Transmission or Distribution System Owner"].str.contains(
        "Virginia Electric|Dominion", case=False, na=False
    )
].copy()

df_dom_eia = df_dom_eia.dropna(subset=["Latitude", "Longitude"])
print(f"Dominion plants with coordinates: {len(df_dom_eia)}")

# Download Virginia state boundary from Census Bureau (free)
print("\nDownloading Virginia boundary...")
va_url = "https://www2.census.gov/geo/tiger/TIGER2022/STATE/tl_2022_us_state.zip"
r2 = requests.get(va_url)
z2 = zipfile.ZipFile(io.BytesIO(r2.content))
z2.extractall("us_states")

# Load state shapefile
states = gpd.read_file("us_states/tl_2022_us_state.shp")
virginia = states[states["NAME"] == "Virginia"]
print(f"Virginia boundary loaded: {virginia.crs}")

# Convert plants to geodataframe
geometry = [Point(lon, lat) for lon, lat in 
            zip(df_dom_eia["Longitude"], df_dom_eia["Latitude"])]
gdf_plants = gpd.GeoDataFrame(df_dom_eia, geometry=geometry, crs="EPSG:4326")

# Make sure CRS matches
virginia = virginia.to_crs("EPSG:4326")

# Spatial join - keep only plants inside Virginia
gdf_va_plants = gpd.sjoin(gdf_plants, virginia[["NAME", "geometry"]], 
                           how="inner", predicate="within")

print(f"\nPlants confirmed inside Virginia: {len(gdf_va_plants)}")
print(f"State breakdown after spatial filter:")
print(gdf_va_plants["State"].value_counts())

print("\nSample Virginia plant names:")
print(gdf_va_plants["Plant Name"].head(20).tolist())

Dominion plants with coordinates: 369

Virginia boundary loaded: EPSG:4269

Plants confirmed inside Virginia: 169
State breakdown after spatial filter:
State
VA    169
Name: count, dtype: int64

Sample Virginia plant names:
['Bremo Bluff', 'Chesterfield', 'Cushaw', 'Low Moor', 'Northern Neck', 'Chesapeake', 'Possum Point', 'Surry', 'Yorktown', 'John H Kerr', 'Bath County', 'North Anna', 'Gravel Neck', 'Darbytown', 'Clover', 'Marsh Run Generation Facility', 'Louisa Generation Facility', 'Remington', 'Ladysmith', 'WestRock-West Point Mill']


In [13]:
import pandas as pd
import re
from thefuzz import fuzz, process

# Load active DOM pnodes
df_dom = pd.read_csv("dom_pnodes.csv")
df_active = df_dom[df_dom["termination_date"].str.startswith("9999")].copy()

# Clean pnode names - remove voltage and special chars
df_active["name_clean"] = (df_active["pnode_name"]
    .str.replace(r'\d+\s*KV.*$', '', regex=True, flags=re.IGNORECASE)
    .str.replace(r'[^A-Z\s]', '', regex=True)
    .str.strip()
)

# Clean EIA plant names
va_plant_names = (gdf_va_plants["Plant Name"]
    .str.upper()
    .str.replace(r'[^A-Z\s]', '', regex=True)
    .str.strip()
    .tolist()
)

print("Sample cleaned pnode names:")
print(df_active["name_clean"].head(10).tolist())
print("\nSample cleaned VA plant names:")
print(va_plant_names[:10])

# Fuzzy match each pnode name against VA plant names
# Only match GEN nodes since EIA tracks generation plants
df_gen = df_active[df_active["pnode_subtype"] == "GEN"].copy()
print(f"\nGEN nodes to match: {len(df_gen)}")

results = []
for _, row in df_gen.iterrows():
    match, score, _ = process.extractOne(
        row["name_clean"],
        va_plant_names,
        scorer=fuzz.token_sort_ratio
    )
    results.append({
        "pnode_id": row["pnode_id"],
        "pnode_name": row["pnode_name"],
        "name_clean": row["name_clean"],
        "best_match": match,
        "score": score
    })

df_matches = pd.DataFrame(results)

# Look at score distribution
print("\nScore distribution:")
print(pd.cut(df_matches["score"], bins=[0,50,70,85,100]).value_counts().sort_index())

# Show high confidence matches
df_good = df_matches[df_matches["score"] >= 80]
print(f"\nHigh confidence matches (score >= 80): {len(df_good)}")
print(df_good[["pnode_name", "best_match", "score"]].to_string())

Sample cleaned pnode names:
['ACCA', 'ACCA', 'ACCA', 'ACCA', 'ACCA', 'ALLIED', 'ALLIED', 'ALLIED', 'ALPINE', 'ALPINE']

Sample cleaned VA plant names:
['BREMO BLUFF', 'CHESTERFIELD', 'CUSHAW', 'LOW MOOR', 'NORTHERN NECK', 'CHESAPEAKE', 'POSSUM POINT', 'SURRY', 'YORKTOWN', 'JOHN H KERR']

GEN nodes to match: 380


ValueError: not enough values to unpack (expected 3, got 2)

In [17]:
import pandas as pd

# Check the type breakdown in our actual pulled data
# Remember we pulled 10 rows earlier - let's pull a bigger sample
import requests

API_KEY = "d14f4a5d0aa6498a84ede0d1d5401e10"
url = "https://api.pjm.com/api/v1/rt_hrl_lmps"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

r = requests.get(url, headers=headers, params={
    "rowCount": 1000,
    "startRow": 1,
    "datetime_beginning_ept": "3-19-2026 00:00 to 3-19-2026 23:00",
    "row_is_current": 1,
    "zone": "DOM",
})

data = r.json()
df = pd.DataFrame(data.get("items", []))

print("Total rows fetched:", len(df))
print("\nNode type breakdown:")
print(df["type"].value_counts())

print("\nSample of each type:")
for t in df["type"].unique():
    print(f"\n--- {t} ---")
    print(df[df["type"] == t][["pnode_id", "pnode_name", "type", "zone"]].head(3))

Total rows fetched: 1000

Node type breakdown:
type
LOAD    817
GEN     156
EHV      27
Name: count, dtype: int64

Sample of each type:

--- LOAD ---
   pnode_id pnode_name  type zone
0  34885183       ACCA  LOAD  DOM
1  34885185       ACCA  LOAD  DOM
2  34885187       ACCA  LOAD  DOM

--- GEN ---
     pnode_id pnode_name type zone
677  34887765   CHESTER4  GEN  DOM
678  34887767   CHESTER4  GEN  DOM
679  34887769   CHESTER4  GEN  DOM

--- EHV ---
     pnode_id    pnode_name type zone
764  35010339        CARSON  EHV  DOM
765  35010341  CHICKAHOMINY  EHV  DOM
766  35010343        CLOVER  EHV  DOM


In [19]:
import requests
import time

def geocode_osm(pnode_name):
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": f"{pnode_name} substation Virginia",
        "format": "json",
        "limit": 1,
        "countrycodes": "us",
    }
    headers = {"User-Agent": "pjm_research_geocoder"}
    r = requests.get(url, params=params, headers=headers, timeout=10)
    print("Raw response:", r.text[:300])
    results = r.json()
    if results:
        return {
            "pnode_name": pnode_name,
            "found": True,
            "lat": float(results[0]["lat"]),
            "lon": float(results[0]["lon"]),
            "display_name": results[0]["display_name"],
        }
    return {"pnode_name": pnode_name, "found": False}

test_name = "ATHENIA"
print(f"Testing: '{test_name}'\n")
result = geocode_osm(test_name)
print(result)


Testing: 'ATHENIA'

Raw response: []
{'pnode_name': 'ATHENIA', 'found': False}


In [30]:
import requests
import os
import pandas as pd

API_KEY = os.environ.get("60a6ac7a3925418cbdaa7fdfec6d2b90")
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

r = requests.get(
    "https://api.pjm.com/api/v1/rt_hrl_lmps",
    headers=headers,
    params={
        "rowCount": 50000,
        "startRow": 1,
        "datetime_beginning_ept": "3-19-2026 00:00 to 3-19-2026 23:00",
        "row_is_current": 1,
        "zone": "DOM",
    }
)

data = r.json()
df = pd.DataFrame(data.get("items", []))

print("Transmission zone unique values:")
print(df["zone"].value_counts())

print("\nSample rows:")
print(df[["pnode_name", "type", "zone"]].head(20))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [34]:
import requests
import pandas as pd
import re

API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"  # paste your key here locally, don't share it in chat
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

r = requests.get(
    "https://api.pjm.com/api/v1/pnode",
    headers=headers,
    params={"rowCount": 50000, "startRow": 1, "zone": "DOM"}
)

print("Status:", r.status_code)
print("Raw:", r.text[:200])

Status: 200
Raw: {"links":[{"rel":"self","href":"https://api.pjm.com/api/v1/pnode?RowCount=50000&Order=Asc&StartRow=1&IsActiveMetadata=True&Fields=effective_date%2Cpnode_id%2Cpnode_name%2Cpnode_subtype%2Cpnode_type%2C


In [39]:
import pandas as pd

API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

r = requests.get(
    "https://api.pjm.com/api/v1/pnode",
    headers=headers,
    params={"rowCount": 50000, "startRow": 1, "zone": "DOM"}
)

data = r.json()
df_dom = pd.DataFrame(data.get("items", []))
df_active = df_dom[df_dom["termination_date"].str.startswith("9999")].copy()

print(f"Total active pnodes: {len(df_active)}")
print(f"Unique pnode names (substations): {df_active['pnode_name'].nunique()}")

print("\nSample unique names:")
print(sorted(df_active["pnode_name"].unique())[:20])

import pandas as pd
import re

# Strip voltage and equipment - everything after the substation name
df_active["substation_name"] = (df_active["pnode_name"]
    .str.replace(r'\s+\d+\s*KV.*$', '', regex=True, flags=re.IGNORECASE)
    .str.strip()
)

print(f"Total active pnodes: {len(df_active)}")
print(f"Unique pnode names (with voltage/equipment): {df_active['pnode_name'].nunique()}")
print(f"Unique substation names (stripped): {df_active['substation_name'].nunique()}")

print("\nSample mapping:")
print(df_active[["pnode_name", "substation_name"]].head(20).to_string())




Total active pnodes: 1755
Unique pnode names (substations): 1755

Sample unique names:
['ACCA    13 KV   TX1', 'ACCA    13 KV   TX2', 'ACCA    35 KV   TX3', 'ACCA    35 KV   TX4', 'ACCA    35 KV   TX5', 'AHOSKIE 35 KV   COLHALSP', 'AHOSKIE 35 KV   TX1', 'AHOSKIE 35 KV   TX2', 'AIRPORT 230 KV  LOAD', 'ALEXANDC13 KV   TX1', 'ALLEGHAN13 KV   LD2', 'ALLENCRK230 KV  LDU', 'ALLENCRK230 KV  LDV', 'ALLENCRK230 KV  LDY', 'ALLENCRK230 KV  LDZ', 'ALLIED  13 KV   TX1', 'ALLIED  13 KV   TX2', 'ALLIED  13 KV   TX3', 'ALPINE  13 KV   TX1', 'ALPINE  13 KV   TX2']
Total active pnodes: 1755
Unique pnode names (with voltage/equipment): 1755
Unique substation names (stripped): 1439

Sample mapping:
                   pnode_name           substation_name
563       ACCA    13 KV   TX1                      ACCA
564       ACCA    13 KV   TX2                      ACCA
565       ACCA    35 KV   TX3                      ACCA
566       ACCA    35 KV   TX4                      ACCA
567       ACCA    35 KV   TX5   

In [40]:
import requests
import pandas as pd

API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

r = requests.get(
    "https://api.pjm.com/api/v1/rt_hrl_lmps",
    headers=headers,
    params={
        "rowCount": 50000,
        "startRow": 1,
        "datetime_beginning_ept": "3-19-2026 00:00 to 3-19-2026 23:00",
        "row_is_current": 1,
        "zone": "DOM",
    }
)

data = r.json()
df = pd.DataFrame(data.get("items", []))

print(f"Total rows: {len(df)}")
print(f"Unique pnode_names: {df['pnode_name'].nunique()}")
print(f"Unique voltages: {df['voltage'].nunique()}")

print("\nSample:")
print(df[["pnode_name", "voltage", "equipment", "type"]].head(10).to_string())

# Get unique substations - pnode_name is already clean
unique_substations = df["pnode_name"].unique()
print(f"\nUnique substation names to geocode: {len(unique_substations)}")
print(sorted(unique_substations)[:20])

Total rows: 42120
Unique pnode_names: 761
Unique voltages: 27

Sample:
  pnode_name voltage equipment  type
0       ACCA   13 KV       TX1  LOAD
1       ACCA   13 KV       TX2  LOAD
2       ACCA   35 KV       TX3  LOAD
3       ACCA   35 KV       TX4  LOAD
4       ACCA   35 KV       TX5  LOAD
5     ALLIED   13 KV       TX1  LOAD
6     ALLIED   13 KV       TX2  LOAD
7     ALLIED   13 KV       TX3  LOAD
8     ALPINE   13 KV       TX1  LOAD
9     ALPINE   13 KV       TX2  LOAD

Unique substation names to geocode: 761
['ACCA', 'AHOSKIE', 'AIRPORT', 'ALEXANDC', 'ALLEGHAN', 'ALLENCRK', 'ALLIED', 'ALPINE', 'ALTAIR', 'ALTAVIS2', 'ALTAVSTA', 'ALTON', 'ANACONDA', 'ANNANDAL', 'AQUIA', 'ARCOLA', 'ARLINGTN', 'ARNOLDCR', 'ASHBURN', 'ATLNTCDP']


In [41]:
import requests
import pandas as pd

API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

# Check all zones available in rt_hrl_lmps
# Pull a sample with no zone filter to see all zones
r = requests.get(
    "https://api.pjm.com/api/v1/rt_hrl_lmps",
    headers=headers,
    params={
        "rowCount": 50000,
        "startRow": 1,
        "datetime_beginning_ept": "3-19-2026 00:00 to 3-19-2026 01:00",  # just 1 hour
        "row_is_current": 1,
    }
)

data = r.json()
df = pd.DataFrame(data.get("items", []))
print(f"Total rows for 1 hour: {len(df)}")
print(f"\nAll zones present:")
print(df["zone"].value_counts())

Total rows for 1 hour: 28746

All zones present:
zone
AEP         5854
DOM         3510
COMED       2850
ATSI        2788
APS         1758
PSEG        1552
PPL         1504
DEOK        1178
PECO         888
EKPC         854
PENELEC      826
DPL          784
DAY          684
JCPL         574
METED        562
BGE          526
AECO         452
PEPCO        440
DUQ          260
DUKE         116
RECO          48
OVEC          36
CPL           32
EXTERNAL      14
Name: count, dtype: int64


In [42]:
import pandas as pd
import zipfile
import io
import requests

# Reload EIA plant data
url = "https://www.eia.gov/electricity/data/eia860/archive/xls/eia8602022.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

with z.open("2___Plant_Y2022.xlsx") as f:
    df_plants = pd.read_excel(f, sheet_name="Plant", header=1)

# Filter to PJM plants in Virginia
df_pjm = df_plants[df_plants["Balancing Authority Code"] == "PJM"]

# Cross-reference zone (transmission owner) with state
print("Zones with plants in Virginia:")
va_zones = (df_pjm[df_pjm["State"] == "VA"]
    ["Transmission or Distribution System Owner"]
    .value_counts()
)
print(va_zones)

print("\nFor comparison, zones with plants in NC:")
nc_zones = (df_pjm[df_pjm["State"] == "NC"]
    ["Transmission or Distribution System Owner"]
    .value_counts()
)
print(nc_zones)

Zones with plants in Virginia:
Transmission or Distribution System Owner
Virginia Electric & Power Co                 158
Appalachian Power Co                          27
Old Dominion Electric Coop                    11
City of Manassas - (VA)                        8
Shenandoah Valley Elec Coop                    5
Town of Bedford - (VA)                         5
A & N Electric Coop                            5
Delmarva Power                                 4
Rappahannock Electric Coop                     4
City of Danville - (VA)                        3
City of Harrisonburg - (VA)                    3
City of Salem - (VA)                           3
City of Danville                               3
City of Martinsville - (VA)                    2
City of Franklin - (VA)                        2
Central Virginia Electric Coop                 2
City of Radford - (VA)                         1
Southside Electric Coop, Inc                   1
Community Electric Coop                      

In [43]:
# Map transmission owners to PJM zone codes
owner_to_zone = {
    "Virginia Electric & Power Co": "DOM",
    "Virginia Electric & Power Co": "DOM",  
    "Appalachian Power Co": "APS",
    "Delmarva Power": "DPL",
    "Potomac Electric Power Co": "PEPCO",
    # Cooperatives - these are within DOM or APS zones
    "Old Dominion Electric Coop": "DOM",
    "Shenandoah Valley Elec Coop": "APS",
    "Rappahannock Electric Coop": "DOM",
    "Northern Virginia Elec Coop": "DOM",
    "A & N Electric Coop": "DPL",
    "Central Virginia Electric Coop": "DOM",
    # Municipal utilities - within DOM zone
    "City of Manassas - (VA)": "DOM",
    "City of Danville - (VA)": "DOM",
    "City of Harrisonburg - (VA)": "DOM",
    "City of Salem - (VA)": "APS",
    "City of Radford - (VA)": "APS",
    "Town of Bedford - (VA)": "APS",
}

print("Virginia PJM zones summary:")
print("""
DOM  (Dominion/Virginia Electric & Power) - covers most of Virginia
APS  (Appalachian Power)                  - covers southwest Virginia  
DPL  (Delmarva Power)                     - covers Eastern Shore of Virginia
PEPCO (Potomac Electric Power)            - covers small part of northern Virginia
""")

# Check how many nodes each Virginia zone has in the LMP data
virginia_zones = ["DOM", "APS", "DPL", "PEPCO"]
print("Node counts per Virginia zone (from 1-hour LMP sample):")
for zone in virginia_zones:
    count = len(df[df["zone"] == zone])
    print(f"  {zone}: {count} nodes")

print(f"\nTotal Virginia nodes: {sum(len(df[df['zone']==z]) for z in virginia_zones)}")
print(f"As % of all PJM nodes: {sum(len(df[df['zone']==z]) for z in virginia_zones)/len(df)*100:.1f}%")

Virginia PJM zones summary:

DOM  (Dominion/Virginia Electric & Power) - covers most of Virginia
APS  (Appalachian Power)                  - covers southwest Virginia  
DPL  (Delmarva Power)                     - covers Eastern Shore of Virginia
PEPCO (Potomac Electric Power)            - covers small part of northern Virginia

Node counts per Virginia zone (from 1-hour LMP sample):
  DOM: 3510 nodes
  APS: 1758 nodes
  DPL: 784 nodes
  PEPCO: 440 nodes

Total Virginia nodes: 6492
As % of all PJM nodes: 22.6%


In [3]:
import geopandas as gpd
import pandas as pd

# Load the shapefile directly - it already has coordinates built in
gdf_subs = gpd.read_file("/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/electricSubstations/Shapefile/electric_substations_hifld_v4_FieldMods.shp")

print(f"Total substations: {len(gdf_subs)}")
print(f"Columns: {gdf_subs.columns.tolist()}")
print(f"CRS: {gdf_subs.crs}")
print(f"\nSample rows:")
print(gdf_subs.head(5))

Total substations: 75329
Columns: ['Name', 'HIFLD_Name', 'HIFLD_ID', 'city', 'City_Prop', 'zip', 'type', 'Type_Prop', 'status', 'Stat_Prop', 'county', 'Count_Prop', 'countyfips', 'StateShort', 'StateFull', 'country', 'latitude', 'longitude', 'naics_code', 'naics_desc', 'source', 'sourcedate', 'val_method', 'val_date', 'lines', 'LineTxt', 'max_volt', 'CurrentMax', 'min_volt', 'CurrentMin', 'max_infer', 'min_infer', 'MinVoltCat', 'geometry']
CRS: EPSG:3857

Sample rows:
                    Name             HIFLD_Name HIFLD_ID          city  \
0          Unknown107655          UNKNOWN107655   107655    SHELL LAKE   
1          Unknown107656          UNKNOWN107656   107656      PRENTICE   
2               Holcombe               HOLCOMBE   107657      HOLCOMBE   
3  South Sheboygan Falls  SOUTH SHEBOYGAN FALLS   107658  LIMA TOWN OF   
4      Chalk Hills Hydro         CHALK HILLS HY   107659       DAGGETT   

      City_Prop    zip        type   Type_Prop      status   Stat_Prop  ...  \
0  

In [4]:
import geopandas as gpd
import pandas as pd
import requests

# Filter to Virginia substations only
gdf_va = gdf_subs[gdf_subs["StateShort"] == "VA"].copy()
print(f"Virginia substations: {len(gdf_va)}")
print(f"\nSample Virginia substations:")
print(gdf_va[["Name", "HIFLD_Name", "city", "county", "max_volt", "latitude", "longitude"]].head(20).to_string())

Virginia substations: 1432

Sample Virginia substations:
                                  Name                        HIFLD_Name            city     county  max_volt   latitude  longitude
979                               Gore                     UNKNOWN108970            GORE  FREDERICK -999999.0  39.276203 -78.364642
1383                           Imboden                           IMBODEN      APPALACHIA       WISE     161.0  36.881393 -82.812629
1391                     Unknown109426                     UNKNOWN109426      JONESVILLE        LEE -999999.0  36.726088 -83.110564
1395                        Dorchester                        DORCHESTER          NORTON     NORTON     161.0  36.932992 -82.646731
1450                     Unknown109487                     UNKNOWN109487       ROSE HILL        LEE      69.0  36.665037 -83.382438
4639                         Richlands                         RICHLANDS       RICHLANDS   TAZEWELL     138.0  37.092140 -81.806643
4640               

In [5]:
import pandas as pd
from thefuzz import fuzz, process

API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

# Get all Virginia zone pnodes from LMP data
r = requests.get(
    "https://api.pjm.com/api/v1/rt_hrl_lmps",
    headers=headers,
    params={
        "rowCount": 50000,
        "startRow": 1,
        "datetime_beginning_ept": "3-19-2026 00:00 to 3-19-2026 01:00",
        "row_is_current": 1,
        "zone": "DOM",  # start with DOM
    }
)

data = r.json()
df_lmp = pd.DataFrame(data.get("items", []))

# Get unique pnode names
unique_pnodes = df_lmp["pnode_name"].unique()
print(f"Unique DOM pnode names: {len(unique_pnodes)}")

# Get Virginia substation names for matching
va_sub_names = gdf_va["Name"].str.upper().tolist()
va_sub_hifld = gdf_va["HIFLD_Name"].tolist()

print(f"Virginia HIFLD substations: {len(va_sub_names)}")

# Match each pnode name to closest Virginia substation
print("\nMatching pnode names to Virginia substations...\n")
results = []
for pnode in unique_pnodes:
    # Try matching against both Name and HIFLD_Name
    match_name, score_name = process.extractOne(
        pnode, va_sub_names, scorer=fuzz.token_sort_ratio
    )
    match_hifld, score_hifld = process.extractOne(
        pnode, va_sub_hifld, scorer=fuzz.token_sort_ratio
    )
    
    # Take the better match
    if score_name >= score_hifld:
        best_match, best_score = match_name, score_name
    else:
        best_match, best_score = match_hifld, score_hifld
    
    results.append({
        "pnode_name": pnode,
        "best_match": best_match,
        "score": best_score
    })

df_matches = pd.DataFrame(results)

# Score distribution
print("Score distribution:")
print(pd.cut(df_matches["score"], 
             bins=[0, 50, 70, 85, 95, 100])
      .value_counts()
      .sort_index())

# Show high confidence matches
df_good = df_matches[df_matches["score"] >= 85]
print(f"\nHigh confidence matches (>=85): {len(df_good)}/{len(df_matches)}")
print(df_good[["pnode_name", "best_match", "score"]].head(20).to_string())

# Show low confidence matches
df_bad = df_matches[df_matches["score"] < 70]
print(f"\nLow confidence matches (<70): {len(df_bad)}/{len(df_matches)}")
print(df_bad[["pnode_name", "best_match", "score"]].head(20).to_string())

Unique DOM pnode names: 761
Virginia HIFLD substations: 1432

Matching pnode names to Virginia substations...

Score distribution:
score
(0, 50]       46
(50, 70]     389
(70, 85]     137
(85, 95]      88
(95, 100]    101
Name: count, dtype: int64

High confidence matches (>=85): 189/761
   pnode_name  best_match  score
0        ACCA        ACCA    100
1      ALLIED      ALLIED    100
2      ALPINE      ALPINE    100
4    BANISTE4    BANISTER     88
6       BASIN       BASIN    100
9    BELLWOOD    BELLWOOD    100
12    BRODNAX     BRODNAX    100
15     CARVER      CARVER    100
16    CENTRAL   CENTRALIA     88
17   CENTRLIA   CENTRALIA     94
19   CHASECTY  CHASE CITY     89
20   CHATHAM4     CHATHAM     93
23      CREWE       CREWE    100
27   DISPUTNT  DISPUTANTA     89
29    DRYBURG     DRYBURG    100
30   DUNNSVIL  DUNNSVILLE     89
31    DUPONT4      DUPONT     92
32    ELMONT4      ELMONT     92
34    EMPORIA     EMPORIA    100
36   FARMVILL   FARMVILLE     94

Low confidence ma

In [6]:
# Search for anything similar in Virginia HIFLD data
print("HIFLD substations with similar names:")
matches = gdf_va[gdf_va["Name"].str.contains("BAKER", case=False, na=False)]
print(matches[["Name", "HIFLD_Name", "city", "county", "latitude", "longitude"]].to_string())

print("\nAlso check broader PJM pnode list for context:")
similar = df_lmp[df_lmp["pnode_name"].str.contains("BAKR", case=False, na=False)]
print(similar[["pnode_name", "voltage", "equipment", "type", "zone"]].to_string())

HIFLD substations with similar names:
Empty DataFrame
Columns: [Name, HIFLD_Name, city, county, latitude, longitude]
Index: []

Also check broader PJM pnode list for context:
     pnode_name voltage equipment  type zone
11     BAKRSPDP  115 KV       TX1  LOAD  DOM
1766   BAKRSPDP  115 KV       TX1  LOAD  DOM


In [7]:
import geopandas as gpd
import pandas as pd

# Load utility territories shapefile
gdf_util = gpd.read_file("/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/electricUtilityTerritories/Shapefile/Electric_Retail_Service_Territories_CustomFields.shp")

print(f"Total utility territories: {len(gdf_util)}")
print(f"Columns: {gdf_util.columns.tolist()}")
print(f"CRS: {gdf_util.crs}")
print(f"\nSample rows:")
print(gdf_util.head(5))

Total utility territories: 2949
Columns: ['OBJECTID', 'ID', 'NAME', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'TYPE', 'COUNTRY', 'SOURCE', 'SOURCEDATE', 'VAL_DATE', 'WEBSITE', 'REGULATED', 'CNTRL_AREA', 'PLAN_AREA', 'HOLDING_CO', 'SUMMR_PEAK', 'WINTR_PEAK', 'SUMMER_CAP', 'WINTER_CAP', 'NET_GEN', 'PURCHASED', 'NET_EX', 'RETAIL_MWH', 'WSALE_MWH', 'TOTAL_MWH', 'TRANS_MWH', 'CUSTOMERS', 'YEAR', 'Shape__Are', 'Shape__Len', 'NameProp', 'TypeProp', 'CityProp', 'CntrlProp', 'SimpleType', 'HoldCoProp', 'NameFinal', 'TypeAdj', 'geometry']
CRS: EPSG:4326

Sample rows:
   OBJECTID     ID                      NAME            ADDRESS  \
0         1   1000    CITY OF AUGUSTA - (AR)      NOT AVAILABLE   
1         2  10000              EVERGY METRO   1200 MAIN STREET   
2         3  10005  EVERGY KANSAS SOUTH, INC  818 KANSAS AVENUE   
3         4  10009  KARNES ELECTRIC COOP INC   1007 N. HWY. 123   
4         5  10012         KAY ELECTRIC COOP      300 W. DOOLIN   

            CITY STATE            ZIP   

In [8]:
import geopandas as gpd

# Reload utility territories fresh
gdf_util = gpd.read_file("/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/electricUtilityTerritories/Shapefile/Electric_Retail_Service_Territories_CustomFields.shp")

# Filter directly on original dataframe before spatial join
# using STATE and CNTRL_AREA fields
va_pjm = gdf_util[
    (gdf_util["CNTRL_AREA"] == "PJM INTERCONNECTION, LLC") &
    (gdf_util["STATE"] == "VA")
]

print(f"Virginia PJM utilities: {len(va_pjm)}")
print(va_pjm[["NAME", "TYPE", "PLAN_AREA", "CNTRL_AREA", "STATE"]].to_string())

Virginia PJM utilities: 28
                                  NAME            TYPE      PLAN_AREA                CNTRL_AREA STATE
93              BARC ELECTRIC COOP INC     COOPERATIVE  NOT AVAILABLE  PJM INTERCONNECTION, LLC    VA
252            CITY OF MANASSAS - (VA)       MUNICIPAL  NOT AVAILABLE  PJM INTERCONNECTION, LLC    VA
287        CITY OF MARTINSVILLE - (VA)       MUNICIPAL  NOT AVAILABLE  PJM INTERCONNECTION, LLC    VA
327   MECKLENBURG ELECTRIC COOPERATIVE     COOPERATIVE  NOT AVAILABLE  PJM INTERCONNECTION, LLC    VA
554        NORTHERN VIRGINIA ELEC COOP     COOPERATIVE  NOT AVAILABLE  PJM INTERCONNECTION, LLC    VA
592       NORTHERN NECK ELEC COOP, INC     COOPERATIVE  NOT AVAILABLE  PJM INTERCONNECTION, LLC    VA
729             TOWN OF BEDFORD - (VA)       MUNICIPAL  NOT AVAILABLE  PJM INTERCONNECTION, LLC    VA
854        PRINCE GEORGE ELECTRIC COOP     COOPERATIVE  NOT AVAILABLE  PJM INTERCONNECTION, LLC    VA
876             CITY OF RADFORD - (VA)       MUNICIPAL 

In [52]:
print("Columns after spatial join:")
print(gdf_util_va.columns.tolist())

Columns after spatial join:
['OBJECTID', 'ID', 'NAME_left', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'TYPE', 'COUNTRY', 'SOURCE', 'SOURCEDATE', 'VAL_DATE', 'WEBSITE', 'REGULATED', 'CNTRL_AREA', 'PLAN_AREA', 'HOLDING_CO', 'SUMMR_PEAK', 'WINTR_PEAK', 'SUMMER_CAP', 'WINTER_CAP', 'NET_GEN', 'PURCHASED', 'NET_EX', 'RETAIL_MWH', 'WSALE_MWH', 'TOTAL_MWH', 'TRANS_MWH', 'CUSTOMERS', 'YEAR', 'Shape__Are', 'Shape__Len', 'NameProp', 'TypeProp', 'CityProp', 'CntrlProp', 'SimpleType', 'HoldCoProp', 'NameFinal', 'TypeAdj', 'geometry', 'index_right', 'NAME_right']


In [53]:
print(f"Utility territories intersecting Virginia: {len(gdf_util_va)}")
print(f"\nUtility names in Virginia:")
print(gdf_util_va[["NAME_left", "TYPE", "CNTRL_AREA", "STATE"]].to_string())

Utility territories intersecting Virginia: 56

Utility names in Virginia:
                               NAME_left            TYPE                                                  CNTRL_AREA STATE
24                 KENTUCKY UTILITIES CO  INVESTOR OWNED  LOUISVILLE GAS AND ELECTRIC COMPANY AND KENTUCKY UTILITIES    KY
43                    KINGSPORT POWER CO  INVESTOR OWNED                                    PJM INTERCONNECTION, LLC    OH
93                BARC ELECTRIC COOP INC     COOPERATIVE                                    PJM INTERCONNECTION, LLC    VA
252              CITY OF MANASSAS - (VA)       MUNICIPAL                                    PJM INTERCONNECTION, LLC    VA
287          CITY OF MARTINSVILLE - (VA)       MUNICIPAL                                    PJM INTERCONNECTION, LLC    VA
327     MECKLENBURG ELECTRIC COOPERATIVE     COOPERATIVE                                    PJM INTERCONNECTION, LLC    VA
407                 MONONGAHELA POWER CO  INVESTOR OWNED         

In [9]:
# Filter to Virginia PJM utilities only
va_pjm_utilities = gdf_util_va[
    (gdf_util_va["CNTRL_AREA"] == "PJM INTERCONNECTION, LLC") &
    (gdf_util_va["STATE"] == "VA")
]["NAME_left"].tolist()

print(f"Virginia PJM utilities: {len(va_pjm_utilities)}")
print(va_pjm_utilities)

# Now filter HIFLD substations to only those owned by Virginia PJM utilities
# First check what owner field exists in HIFLD data
print("\nHIFLD substation columns related to owner:")
print(gdf_subs[["Name", "HIFLD_Name", "StateShort"]].head(5))

# Filter Virginia substations that are in PJM
gdf_va_pjm = gdf_subs[gdf_subs["StateShort"] == "VA"].copy()
print(f"\nAll Virginia HIFLD substations: {len(gdf_va_pjm)}")

# Now match these to pnode names
va_sub_names = gdf_va_pjm["HIFLD_Name"].str.upper().tolist()
print(f"Virginia substation names for matching: {len(va_sub_names)}")
print("\nSample names:")
print(sorted(va_sub_names)[:20])

NameError: name 'gdf_util_va' is not defined

In [55]:
# Filter to Virginia PJM utilities - investor owned and cooperative only
va_pjm_filtered = gdf_util_va[
    (gdf_util_va["CNTRL_AREA"] == "PJM INTERCONNECTION, LLC") &
    (gdf_util_va["STATE"] == "VA") &
    (gdf_util_va["TYPE"].isin(["INVESTOR OWNED", "COOPERATIVE"]))
]

print(f"Virginia PJM investor owned and cooperative utilities: {len(va_pjm_filtered)}")
print(va_pjm_filtered[["NAME_left", "TYPE", "PLAN_AREA"]].to_string())

Virginia PJM investor owned and cooperative utilities: 13
                             NAME_left            TYPE      PLAN_AREA
93              BARC ELECTRIC COOP INC     COOPERATIVE  NOT AVAILABLE
327   MECKLENBURG ELECTRIC COOPERATIVE     COOPERATIVE  NOT AVAILABLE
554        NORTHERN VIRGINIA ELEC COOP     COOPERATIVE  NOT AVAILABLE
592       NORTHERN NECK ELEC COOP, INC     COOPERATIVE  NOT AVAILABLE
854        PRINCE GEORGE ELECTRIC COOP     COOPERATIVE  NOT AVAILABLE
1076       SHENANDOAH VALLEY ELEC COOP     COOPERATIVE  NOT AVAILABLE
1461      VIRGINIA ELECTRIC & POWER CO  INVESTOR OWNED  NOT AVAILABLE
1659      SOUTHSIDE ELECTRIC COOP, INC     COOPERATIVE  NOT AVAILABLE
1875    CENTRAL VIRGINIA ELECTRIC COOP     COOPERATIVE  NOT AVAILABLE
1993        RAPPAHANNOCK ELECTRIC COOP     COOPERATIVE  NOT AVAILABLE
2039           COMMUNITY ELECTRIC COOP     COOPERATIVE  NOT AVAILABLE
2083     CRAIG-BOTETOURT ELECTRIC COOP     COOPERATIVE  NOT AVAILABLE
2615               A & N ELECTRI

In [58]:
va_pjm = gdf_util_va[
    (gdf_util_va["CNTRL_AREA"] == "PJM INTERCONNECTION, LLC") &
    (gdf_util_va["STATE"] == "VA") &
    (gdf_util_va["TYPE"].isin(["INVESTOR OWNED", "COOPERATIVE"]))
]

print(f"Virginia PJM utilities: {len(va_pjm)}")
print(va_pjm[["NAME_left", "TYPE",  "CNTRL_AREA", "STATE"]].to_string())

Virginia PJM utilities: 13
                             NAME_left            TYPE                CNTRL_AREA STATE
93              BARC ELECTRIC COOP INC     COOPERATIVE  PJM INTERCONNECTION, LLC    VA
327   MECKLENBURG ELECTRIC COOPERATIVE     COOPERATIVE  PJM INTERCONNECTION, LLC    VA
554        NORTHERN VIRGINIA ELEC COOP     COOPERATIVE  PJM INTERCONNECTION, LLC    VA
592       NORTHERN NECK ELEC COOP, INC     COOPERATIVE  PJM INTERCONNECTION, LLC    VA
854        PRINCE GEORGE ELECTRIC COOP     COOPERATIVE  PJM INTERCONNECTION, LLC    VA
1076       SHENANDOAH VALLEY ELEC COOP     COOPERATIVE  PJM INTERCONNECTION, LLC    VA
1461      VIRGINIA ELECTRIC & POWER CO  INVESTOR OWNED  PJM INTERCONNECTION, LLC    VA
1659      SOUTHSIDE ELECTRIC COOP, INC     COOPERATIVE  PJM INTERCONNECTION, LLC    VA
1875    CENTRAL VIRGINIA ELECTRIC COOP     COOPERATIVE  PJM INTERCONNECTION, LLC    VA
1993        RAPPAHANNOCK ELECTRIC COOP     COOPERATIVE  PJM INTERCONNECTION, LLC    VA
2039           C

In [14]:
import geopandas as gpd

# Load utility territories
gdf_util = gpd.read_file("/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/electricUtilityTerritories/Shapefile/Electric_Retail_Service_Territories_CustomFields.shp")

# Load Virginia boundary and match CRS
states = gpd.read_file("us_states/tl_2022_us_state.shp")
virginia = states[states["NAME"] == "Virginia"].to_crs(gdf_util.crs)

# Spatial join
gdf_va = gpd.sjoin(
    gdf_util,
    virginia[["NAME", "geometry"]],
    how="inner",
    predicate="intersects"
)

# Filter to PJM only
va_pjm = gdf_va[gdf_va["CNTRL_AREA"] == "PJM INTERCONNECTION, LLC"]

print(f"Utilities intersecting Virginia: {len(gdf_va)}")
print(f"Virginia PJM utilities: {len(va_pjm)}")
print(va_pjm[["NAME_left", "TYPE", "CNTRL_AREA", "STATE", "PLAN_AREA"]].to_string())

Utilities intersecting Virginia: 56
Virginia PJM utilities: 42
                               NAME_left            TYPE                CNTRL_AREA STATE                PLAN_AREA
43                    KINGSPORT POWER CO  INVESTOR OWNED  PJM INTERCONNECTION, LLC    OH            NOT AVAILABLE
93                BARC ELECTRIC COOP INC     COOPERATIVE  PJM INTERCONNECTION, LLC    VA            NOT AVAILABLE
252              CITY OF MANASSAS - (VA)       MUNICIPAL  PJM INTERCONNECTION, LLC    VA            NOT AVAILABLE
287          CITY OF MARTINSVILLE - (VA)       MUNICIPAL  PJM INTERCONNECTION, LLC    VA            NOT AVAILABLE
327     MECKLENBURG ELECTRIC COOPERATIVE     COOPERATIVE  PJM INTERCONNECTION, LLC    VA            NOT AVAILABLE
407                 MONONGAHELA POWER CO  INVESTOR OWNED  PJM INTERCONNECTION, LLC    PA            NOT AVAILABLE
554          NORTHERN VIRGINIA ELEC COOP     COOPERATIVE  PJM INTERCONNECTION, LLC    VA            NOT AVAILABLE
592         NORTHERN NECK

In [13]:
print(gdf_va.columns.tolist())

['Name', 'HIFLD_Name', 'HIFLD_ID', 'city', 'City_Prop', 'zip', 'type', 'Type_Prop', 'status', 'Stat_Prop', 'county', 'Count_Prop', 'countyfips', 'StateShort', 'StateFull', 'country', 'latitude', 'longitude', 'naics_code', 'naics_desc', 'source', 'sourcedate', 'val_method', 'val_date', 'lines', 'LineTxt', 'max_volt', 'CurrentMax', 'min_volt', 'CurrentMin', 'max_infer', 'min_infer', 'MinVoltCat', 'geometry']


In [15]:
# Get substations within DOM, AEP, and VA coops
target_utilities = gdf_va[
    (gdf_va["CNTRL_AREA"] == "PJM INTERCONNECTION, LLC") &
    (
        (gdf_va["NAME_left"].isin([
            "VIRGINIA ELECTRIC & POWER CO",
            "APPALACHIAN POWER CO"
        ])) |
        ((gdf_va["TYPE"] == "COOPERATIVE") & (gdf_va["STATE"] == "VA"))
    )
]

# Clip to Virginia
target_clipped = gpd.clip(target_utilities, virginia)

# Load HIFLD substations
gdf_subs = gpd.read_file("/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/electricSubstations/Shapefile/electric_substations_hifld_v4_FieldMods.shp")
gdf_subs_va = gdf_subs[gdf_subs["StateShort"] == "VA"].copy()
gdf_subs_va = gdf_subs_va.to_crs(target_clipped.crs)

# Spatial join - keep only substations within target utility territories
subs_in_target = gpd.sjoin(
    gdf_subs_va,
    target_clipped[["NAME_left", "TYPE", "geometry"]],
    how="inner",
    predicate="within"
)

print(f"Total Virginia substations: {len(gdf_subs_va)}")
print(f"Substations in target utilities: {len(subs_in_target)}")
print(f"\nBreakdown by utility:")
print(subs_in_target["NAME_left"].value_counts())
print(f"\nSample substations:")
print(subs_in_target[["Name", "HIFLD_Name", "city", "county", 
                        "max_volt", "latitude", "longitude", 
                        "NAME_left"]].head(20).to_string())

Total Virginia substations: 1432
Substations in target utilities: 1825

Breakdown by utility:
NAME_left
VIRGINIA ELECTRIC & POWER CO        786
APPALACHIAN POWER CO                365
RAPPAHANNOCK ELECTRIC COOP          114
SOUTHSIDE ELECTRIC COOP, INC         95
NORTHERN VIRGINIA ELEC COOP          93
CRAIG-BOTETOURT ELECTRIC COOP        78
MECKLENBURG ELECTRIC COOPERATIVE     69
SHENANDOAH VALLEY ELEC COOP          64
CENTRAL VIRGINIA ELECTRIC COOP       53
COMMUNITY ELECTRIC COOP              30
A & N ELECTRIC COOP                  27
BARC ELECTRIC COOP INC               19
NORTHERN NECK ELEC COOP, INC         18
PRINCE GEORGE ELECTRIC COOP          14
Name: count, dtype: int64

Sample substations:
                                  Name                        HIFLD_Name            city     county  max_volt   latitude  longitude                     NAME_left
979                               Gore                     UNKNOWN108970            GORE  FREDERICK -999999.0  39.276203 -78.36

In [16]:
# Spatial join - keep only substations within target utility territories
subs_in_target = gpd.sjoin(
    gdf_subs_va,
    target_clipped[["NAME_left", "TYPE", "geometry"]],
    how="inner",
    predicate="within"
).drop_duplicates(subset="HIFLD_ID")

print(f"Total Virginia substations: {len(gdf_subs_va)}")
print(f"Substations in DOM, AEP, and VA coops: {len(subs_in_target)}")

print(f"\nBreakdown by utility:")
print(subs_in_target["NAME_left"].value_counts())

print(f"\nSample substations:")
print(subs_in_target[["Name", "HIFLD_Name", "city", "county",
                       "max_volt", "latitude", "longitude",
                       "NAME_left"]].head(20).to_string())

Total Virginia substations: 1432
Substations in DOM, AEP, and VA coops: 1355

Breakdown by utility:
NAME_left
VIRGINIA ELECTRIC & POWER CO        524
APPALACHIAN POWER CO                252
RAPPAHANNOCK ELECTRIC COOP          114
NORTHERN VIRGINIA ELEC COOP          93
SOUTHSIDE ELECTRIC COOP, INC         92
CRAIG-BOTETOURT ELECTRIC COOP        70
MECKLENBURG ELECTRIC COOPERATIVE     69
COMMUNITY ELECTRIC COOP              29
CENTRAL VIRGINIA ELECTRIC COOP       28
SHENANDOAH VALLEY ELEC COOP          27
A & N ELECTRIC COOP                  27
NORTHERN NECK ELEC COOP, INC         18
PRINCE GEORGE ELECTRIC COOP           6
BARC ELECTRIC COOP INC                6
Name: count, dtype: int64

Sample substations:
                                  Name                        HIFLD_Name            city     county  max_volt   latitude  longitude                     NAME_left
979                               Gore                     UNKNOWN108970            GORE  FREDERICK -999999.0  39.276203 

In [17]:
# Check which coops are in APS territory
# Get APS territory
aps_territory = gdf_va[gdf_va["NAME_left"] == "MONONGAHELA POWER CO"]

if len(aps_territory) == 0:
    print("Monongahela Power not found, trying other names...")
    print(gdf_va[gdf_va["TYPE"] == "INVESTOR OWNED"]["NAME_left"].unique())

# Spatially join VA coops to investor owned territories
# to see which major utility each coop sits within
va_coops = gdf_va[
    (gdf_va["TYPE"] == "COOPERATIVE") &
    (gdf_va["STATE"] == "VA") &
    (gdf_va["CNTRL_AREA"] == "PJM INTERCONNECTION, LLC")
].copy()

investor_owned = gdf_va[
    gdf_va["NAME_left"].isin([
        "VIRGINIA ELECTRIC & POWER CO",
        "APPALACHIAN POWER CO",
        "MONONGAHELA POWER CO"
    ])
].copy()

# Spatial join coops to investor owned territories
coops_joined = gpd.sjoin(
    va_coops[["NAME_left", "geometry"]].rename(columns={"NAME_left": "coop_name"}),
    investor_owned[["NAME_left", "geometry"]].rename(columns={"NAME_left": "utility_name"}),
    how="left",
    predicate="intersects"
)

print("Coop to utility mapping:")
print(coops_joined[["coop_name", "utility_name"]].to_string())

Coop to utility mapping:
                             coop_name                  utility_name
93              BARC ELECTRIC COOP INC  VIRGINIA ELECTRIC & POWER CO
93              BARC ELECTRIC COOP INC          APPALACHIAN POWER CO
93              BARC ELECTRIC COOP INC          MONONGAHELA POWER CO
327   MECKLENBURG ELECTRIC COOPERATIVE  VIRGINIA ELECTRIC & POWER CO
327   MECKLENBURG ELECTRIC COOPERATIVE          APPALACHIAN POWER CO
554        NORTHERN VIRGINIA ELEC COOP  VIRGINIA ELECTRIC & POWER CO
592       NORTHERN NECK ELEC COOP, INC  VIRGINIA ELECTRIC & POWER CO
854        PRINCE GEORGE ELECTRIC COOP  VIRGINIA ELECTRIC & POWER CO
1076       SHENANDOAH VALLEY ELEC COOP  VIRGINIA ELECTRIC & POWER CO
1076       SHENANDOAH VALLEY ELEC COOP          MONONGAHELA POWER CO
1659      SOUTHSIDE ELECTRIC COOP, INC  VIRGINIA ELECTRIC & POWER CO
1659      SOUTHSIDE ELECTRIC COOP, INC          APPALACHIAN POWER CO
1875    CENTRAL VIRGINIA ELECTRIC COOP  VIRGINIA ELECTRIC & POWER CO
1875    C

In [18]:
import geopandas as gpd

# Get DOM, AEP, APS utility territories that intersect Virginia
target_investor = gdf_va[
    (gdf_va["CNTRL_AREA"] == "PJM INTERCONNECTION, LLC") &
    (gdf_va["NAME_left"].isin([
        "VIRGINIA ELECTRIC & POWER CO",  # DOM
        "APPALACHIAN POWER CO",           # AEP
        "MONONGAHELA POWER CO"            # APS
    ]))
]

print(f"Target utility territories: {len(target_investor)}")
print(target_investor[["NAME_left", "TYPE", "STATE"]].to_string())

# Clip to Virginia
target_clipped = gpd.clip(target_investor, virginia)

# Load HIFLD substations for Virginia
gdf_subs_va = gdf_subs[gdf_subs["StateShort"] == "VA"].copy()
gdf_subs_va = gdf_subs_va.to_crs(target_clipped.crs)

# Find substations within these utility territories
subs_dom_aep_aps = gpd.sjoin(
    gdf_subs_va,
    target_clipped[["NAME_left", "geometry"]],
    how="inner",
    predicate="within"
).drop_duplicates(subset="HIFLD_ID")

print(f"\nTotal Virginia substations: {len(gdf_subs_va)}")
print(f"Substations in DOM, AEP, APS: {len(subs_dom_aep_aps)}")

print(f"\nBreakdown by utility:")
print(subs_dom_aep_aps["NAME_left"].value_counts())

print(f"\nSample substations:")
print(subs_dom_aep_aps[["Name", "HIFLD_Name", "city", "county",
                          "max_volt", "latitude", "longitude",
                          "NAME_left"]].head(20).to_string())

Target utility territories: 3
                         NAME_left            TYPE STATE
407           MONONGAHELA POWER CO  INVESTOR OWNED    PA
1461  VIRGINIA ELECTRIC & POWER CO  INVESTOR OWNED    VA
2481          APPALACHIAN POWER CO  INVESTOR OWNED    OH

Total Virginia substations: 1432
Substations in DOM, AEP, APS: 1128

Breakdown by utility:
NAME_left
VIRGINIA ELECTRIC & POWER CO    786
APPALACHIAN POWER CO            342
Name: count, dtype: int64

Sample substations:
                                  Name                        HIFLD_Name            city     county  max_volt   latitude  longitude                     NAME_left
4639                         Richlands                         RICHLANDS       RICHLANDS   TAZEWELL     138.0  37.092140 -81.806643  VIRGINIA ELECTRIC & POWER CO
4640                     Claypool Hill                     CLAYPOOL HILL     CEDAR BLUFF   TAZEWELL     138.0  37.045245 -81.775464  VIRGINIA ELECTRIC & POWER CO
4642                       Sandy Ri

In [19]:
# Start with all Virginia substations
gdf_subs_va = gdf_subs[gdf_subs["StateShort"] == "VA"].copy()
gdf_subs_va = gdf_subs_va.to_crs(target_clipped.crs)

# Get all PJM utility territories intersecting Virginia
va_pjm_territories = gdf_va[
    gdf_va["CNTRL_AREA"] == "PJM INTERCONNECTION, LLC"
].copy()

# Spatial join - assign each substation to a utility territory
subs_with_utility = gpd.sjoin(
    gdf_subs_va,
    va_pjm_territories[["NAME_left", "TYPE", "geometry"]],
    how="left",  # keep all substations even if no match
    predicate="within"
).drop_duplicates(subset="HIFLD_ID")

print(f"Total Virginia substations: {len(gdf_subs_va)}")
print(f"Substations with utility assigned: {subs_with_utility['NAME_left'].notna().sum()}")
print(f"Substations with no utility match: {subs_with_utility['NAME_left'].isna().sum()}")

print(f"\nBreakdown by utility:")
print(subs_with_utility["NAME_left"].value_counts())

print(f"\nSample:")
print(subs_with_utility[["Name", "HIFLD_Name", "city", "county",
                           "latitude", "longitude",
                           "NAME_left", "TYPE"]].head(20).to_string())

Total Virginia substations: 1432
Substations with utility assigned: 1379
Substations with no utility match: 53

Breakdown by utility:
NAME_left
VIRGINIA ELECTRIC & POWER CO        570
APPALACHIAN POWER CO                215
RAPPAHANNOCK ELECTRIC COOP          105
NORTHERN VIRGINIA ELEC COOP          93
MECKLENBURG ELECTRIC COOPERATIVE     69
SHENANDOAH VALLEY ELEC COOP          64
CITY OF RADFORD - (VA)               44
CRAIG-BOTETOURT ELECTRIC COOP        30
CITY OF SALEM - (VA)                 29
CENTRAL VIRGINIA ELECTRIC COOP       28
A & N ELECTRIC COOP                  27
BARC ELECTRIC COOP INC               19
NORTHERN NECK ELEC COOP, INC         18
PRINCE GEORGE ELECTRIC COOP          14
CITY OF MANASSAS - (VA)              12
CITY OF HARRISONBURG - (VA)          12
SOUTHSIDE ELECTRIC COOP, INC         11
CITY OF MARTINSVILLE - (VA)           9
CITY OF DANVILLE - (VA)               4
TOWN OF RICHLANDS - (VA)              3
TOWN OF BEDFORD - (VA)                3
Name: count, dty

In [20]:
# Check what PJM zone each coop actually falls under
# by looking at which investor owned territory they overlap with most

# Get just the three major investor owned territories clipped to Virginia
dom = gpd.clip(gdf_va[gdf_va["NAME_left"] == "VIRGINIA ELECTRIC & POWER CO"], virginia)
aep = gpd.clip(gdf_va[gdf_va["NAME_left"] == "APPALACHIAN POWER CO"], virginia)
aps = gpd.clip(gdf_va[gdf_va["NAME_left"] == "MONONGAHELA POWER CO"], virginia)

print(f"DOM territory area in Virginia: {dom.geometry.area.sum():.2f}")
print(f"AEP territory area in Virginia: {aep.geometry.area.sum():.2f}")
print(f"APS territory area in Virginia: {aps.geometry.area.sum():.2f}")

# Check how many Virginia substations fall within each
for name, territory in [("DOM", dom), ("AEP", aep), ("APS", aps)]:
    subs = gpd.sjoin(
        gdf_subs_va,
        territory[["geometry"]],
        how="inner",
        predicate="within"
    ).drop_duplicates(subset="HIFLD_ID")
    print(f"\nSubstations within {name} territory: {len(subs)}")
    print(subs[["Name", "city", "county"]].head(5).to_string())

DOM territory area in Virginia: 5.13
AEP territory area in Virginia: 2.92
APS territory area in Virginia: 0.00

Substations within DOM territory: 786
               Name            city    county
4639      Richlands       RICHLANDS  TAZEWELL
4640  Claypool Hill     CEDAR BLUFF  TAZEWELL
4642    Sandy Ridge       RICHLANDS  TAZEWELL
4643   Jewell Ridge    JEWELL RIDGE  TAZEWELL
4646       Buckhorn  NORTH TAZEWELL  TAZEWELL

Substations within AEP territory: 365
               Name          city    county
4639      Richlands     RICHLANDS  TAZEWELL
4640  Claypool Hill   CEDAR BLUFF  TAZEWELL
4642    Sandy Ridge     RICHLANDS  TAZEWELL
4643   Jewell Ridge  JEWELL RIDGE  TAZEWELL
4644  Jewell Branch  JEWELL RIDGE  BUCHANAN

Substations within APS territory: 0
Empty DataFrame
Columns: [Name, city, county]
Index: []


/var/folders/fk/jrs5sj2165v98_klcg800pk40000gn/T/ipykernel_5499/4268206475.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  print(f"DOM territory area in Virginia: {dom.geometry.area.sum():.2f}")
/var/folders/fk/jrs5sj2165v98_klcg800pk40000gn/T/ipykernel_5499/4268206475.py:10: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  print(f"AEP territory area in Virginia: {aep.geometry.area.sum():.2f}")
/var/folders/fk/jrs5sj2165v98_klcg800pk40000gn/T/ipykernel_5499/4268206475.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  print(f"APS territory area in Virginia: {aps.geometry.area.sum

In [21]:
# Check the coops that appeared to overlap with APS
# Use 'within' instead of 'intersects' to be strict
problem_coops = [
    "BARC ELECTRIC COOP INC",
    "SHENANDOAH VALLEY ELEC COOP", 
    "CRAIG-BOTETOURT ELECTRIC COOP"
]

aps_full = gdf_va[gdf_va["NAME_left"] == "MONONGAHELA POWER CO"]

for coop_name in problem_coops:
    coop = gdf_va[gdf_va["NAME_left"] == coop_name]
    
    # Check intersection area with APS in Virginia
    coop_va = gpd.clip(coop, virginia)
    aps_va = gpd.clip(aps_full, virginia)
    
    if len(aps_va) > 0 and len(coop_va) > 0:
        intersection = gpd.overlay(coop_va, aps_va, how="intersection")
        area = intersection.geometry.area.sum()
    else:
        area = 0
        
    print(f"{coop_name} overlap with APS in Virginia: {area:.4f}")

BARC ELECTRIC COOP INC overlap with APS in Virginia: 0.0000
SHENANDOAH VALLEY ELEC COOP overlap with APS in Virginia: 0.0000
CRAIG-BOTETOURT ELECTRIC COOP overlap with APS in Virginia: 0.0002


/Users/elenamurray/miniconda3/envs/mds_thesis/lib/python3.11/site-packages/geopandas/tools/overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 1 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)
/var/folders/fk/jrs5sj2165v98_klcg800pk40000gn/T/ipykernel_5499/2912987952.py:20: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  area = intersection.geometry.area.sum()
/var/folders/fk/jrs5sj2165v98_klcg800pk40000gn/T/ipykernel_5499/2912987952.py:20: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  area = intersection.geometry.area.sum()
/Users/elenamurray/miniconda3/envs/m

In [22]:
import requests
import pandas as pd
from thefuzz import fuzz, process

API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

# Step 1: Get all pnodes from DOM and AEP zones
all_pnodes = []
for zone in ["DOM", "AEP"]:
    r = requests.get(
        "https://api.pjm.com/api/v1/rt_hrl_lmps",
        headers=headers,
        params={
            "rowCount": 50000,
            "startRow": 1,
            "datetime_beginning_ept": "3-19-2026 00:00 to 3-19-2026 01:00",
            "row_is_current": 1,
            "zone": zone,
        }
    )
    df_zone = pd.DataFrame(r.json().get("items", []))
    all_pnodes.append(df_zone)
    print(f"{zone}: {df_zone['pnode_name'].nunique()} unique pnodes")

df_all = pd.concat(all_pnodes, ignore_index=True)
unique_pnodes = df_all[["pnode_name", "zone"]].drop_duplicates(subset="pnode_name")
print(f"\nTotal unique pnode names: {len(unique_pnodes)}")

# Step 2: Get Virginia substations
gdf_subs_va = gdf_subs[gdf_subs["StateShort"] == "VA"].copy()
gdf_subs_va = gdf_subs_va.to_crs(virginia.crs)

# Clip to DOM and AEP territories only
dom_aep = gdf_va[
    gdf_va["NAME_left"].isin([
        "VIRGINIA ELECTRIC & POWER CO",
        "APPALACHIAN POWER CO"
    ])
].copy()
dom_aep_clipped = gpd.clip(dom_aep, virginia)

# Get substations within DOM and AEP territories
subs_dom_aep = gpd.sjoin(
    gdf_subs_va,
    dom_aep_clipped[["NAME_left", "geometry"]],
    how="inner",
    predicate="within"
).drop_duplicates(subset="HIFLD_ID")

print(f"\nVirginia substations in DOM and AEP: {len(subs_dom_aep)}")
print(subs_dom_aep["NAME_left"].value_counts())

# Step 3: Fuzzy match pnodes to substations
sub_names = subs_dom_aep["HIFLD_Name"].str.upper().tolist()

results = []
for _, row in unique_pnodes.iterrows():
    match, score = process.extractOne(
        row["pnode_name"],
        sub_names,
        scorer=fuzz.token_sort_ratio
    )
    matched_sub = subs_dom_aep[
        subs_dom_aep["HIFLD_Name"].str.upper() == match
    ]
    if len(matched_sub) > 0 and score >= 80:
        sub = matched_sub.iloc[0]
        lat = sub["latitude"]
        lon = sub["longitude"]
        county = sub["county"]
        utility = sub["NAME_left"]
    else:
        lat = lon = county = utility = None

    results.append({
        "pnode_name": row["pnode_name"],
        "zone": row["zone"],
        "best_match": match,
        "score": score,
        "latitude": lat,
        "longitude": lon,
        "county": county,
        "utility": utility
    })

df_matches = pd.DataFrame(results)

print("\nScore distribution:")
print(pd.cut(df_matches["score"],
             bins=[0, 50, 70, 85, 95, 100])
      .value_counts()
      .sort_index())

df_good = df_matches[df_matches["score"] >= 80]
print(f"\nMatched pnodes (score >= 80): {len(df_good)}/{len(df_matches)}")
print(df_good[["pnode_name", "zone", "best_match",
               "score", "county", "utility"]].head(20).to_string())

DOM: 761 unique pnodes
AEP: 1700 unique pnodes

Total unique pnode names: 2461

Virginia substations in DOM and AEP: 1128
NAME_left
VIRGINIA ELECTRIC & POWER CO    786
APPALACHIAN POWER CO            342
Name: count, dtype: int64

Score distribution:
score
(0, 50]       179
(50, 70]     1620
(70, 85]      381
(85, 95]      139
(95, 100]     142
Name: count, dtype: int64

Matched pnodes (score >= 80): 378/2461
   pnode_name zone  best_match  score         county                       utility
0        ACCA  DOM        ACCA    100        HENRICO  VIRGINIA ELECTRIC & POWER CO
1      ALLIED  DOM      ALLIED    100   CHESTERFIELD  VIRGINIA ELECTRIC & POWER CO
2      ALPINE  DOM      ALPINE    100   CHESTERFIELD  VIRGINIA ELECTRIC & POWER CO
4    BANISTE4  DOM    BANISTER     88   PITTSYLVANIA  VIRGINIA ELECTRIC & POWER CO
6       BASIN  DOM       BASIN    100       RICHMOND  VIRGINIA ELECTRIC & POWER CO
9    BELLWOOD  DOM    BELLWOOD    100   CHESTERFIELD  VIRGINIA ELECTRIC & POWER CO
12    

In [23]:
# Check how many AEP pnodes matched to Virginia substations
print("Matched pnodes by zone:")
print(df_good["zone"].value_counts())

print(f"\nUnmatched pnodes by zone:")
df_bad = df_matches[df_matches["score"] < 80]
print(df_bad["zone"].value_counts())

# Most AEP pnodes are likely NOT in Virginia
# Let's check what the unmatched AEP pnodes look like
print("\nSample unmatched AEP pnodes:")
print(df_bad[df_bad["zone"] == "AEP"]["pnode_name"].head(30).tolist())

print("\nSample unmatched DOM pnodes:")
print(df_bad[df_bad["zone"] == "DOM"]["pnode_name"].head(30).tolist())

Matched pnodes by zone:
zone
DOM    209
AEP    169
Name: count, dtype: int64

Unmatched pnodes by zone:
zone
AEP    1531
DOM     552
Name: count, dtype: int64

Sample unmatched AEP pnodes:
['BELPRE', 'CORNER', 'DEGUSSA', 'DUCKCREE', 'ELKEM', 'GOODRICH', 'HARMARHI', 'LAYMAN', 'PORTERFI', 'RENO', 'SHELL', 'WOLFCREE', '18THSTHT', '23RDSTRE', 'ABERT', 'ALLENAEP', 'ALLOY', 'AMALLOY', 'AMOS', 'APPLECRE', 'APPLEVAL', 'ASHEVANS', 'ASHOIL1', 'ASTOR', 'ATKINS', 'AUBURNC3', 'AUBURNC5', 'AUSTINVI', 'BAILEYSV', 'BANCROFT']

Sample unmatched DOM pnodes:
['BAKRSPDP', 'BARNJNDP', 'BEECHDP', 'BELFLDDP', 'BOYDTNDP', 'BRINKDP', 'BROWNBOV', 'BUCKNGHM', 'CHAPARAL', 'CHESTER4', 'CLIMAXDP', 'CRSTLHDP', 'CURDSVDP', 'DANLTNDP', 'DOMINION', 'ENON4', 'FORTLEE', 'FOURRIVR', 'FREEMNDP', 'FTPICKET', 'GARNERDP', 'GARYDP', 'GASBRGDP', 'GREENWDP', 'GRETNA', 'GRETNADP', 'GRITDP', 'HARMONYV', 'HICKGRDP', 'HOOPERDP']


In [24]:
import re

def clean_pnode(name):
    """Strip DP and other common suffixes for better matching."""
    name = name.strip()
    # Remove trailing DP, DPP, 4, etc.
    name = re.sub(r'DP$', '', name).strip()
    name = re.sub(r'DPP$', '', name).strip()
    name = re.sub(r'\d+$', '', name).strip()
    return name

# Rerun matching with cleaned names
results = []
for _, row in unique_pnodes.iterrows():
    cleaned = clean_pnode(row["pnode_name"])
    
    match, score = process.extractOne(
        cleaned,
        sub_names,
        scorer=fuzz.token_sort_ratio
    )
    
    matched_sub = subs_dom_aep[
        subs_dom_aep["HIFLD_Name"].str.upper() == match
    ]
    if len(matched_sub) > 0 and score >= 80:
        sub = matched_sub.iloc[0]
        lat = sub["latitude"]
        lon = sub["longitude"]
        county = sub["county"]
        utility = sub["NAME_left"]
    else:
        lat = lon = county = utility = None

    results.append({
        "pnode_name": row["pnode_name"],
        "pnode_clean": cleaned,
        "zone": row["zone"],
        "best_match": match,
        "score": score,
        "latitude": lat,
        "longitude": lon,
        "county": county,
        "utility": utility
    })

df_matches2 = pd.DataFrame(results)

print("Score distribution after cleaning:")
print(pd.cut(df_matches2["score"],
             bins=[0, 50, 70, 85, 95, 100])
      .value_counts()
      .sort_index())

df_good2 = df_matches2[df_matches2["score"] >= 80]
print(f"\nMatched pnodes (score >= 80): {len(df_good2)}/{len(df_matches2)}")
print(f"\nBy zone:")
print(df_good2["zone"].value_counts())

print("\nSample newly matched DOM pnodes:")
newly_matched = df_good2[
    ~df_good2["pnode_name"].isin(df_good["pnode_name"]) &
    (df_good2["zone"] == "DOM")
]
print(newly_matched[["pnode_name", "pnode_clean", "best_match",
                       "score", "county"]].head(20).to_string())

Score distribution after cleaning:
score
(0, 50]       154
(50, 70]     1614
(70, 85]      400
(85, 95]      130
(95, 100]     163
Name: count, dtype: int64

Matched pnodes (score >= 80): 400/2461

By zone:
zone
DOM    223
AEP    177
Name: count, dtype: int64

Sample newly matched DOM pnodes:
    pnode_name pnode_clean  best_match  score          county
21    CHESTER4     CHESTER  MANCHESTER     82        RICHMOND
39    FREEMNDP      FREEMN    FREEMONT     86       DICKENSON
41    GARNERDP      GARNER   GARNER DP     80  NORTHUMBERLAND
44    GREENWDP      GREENW    GREENWAY     86         LOUDOUN
252   HARRSNDP      HARRSN    HARRISON     86         FAIRFAX
310      COXDP         COX          OX     80         FAIRFAX
324   MARTINDP      MARTIN      MARVIN     83        BUCHANAN
350    BRANDDP       BRAND       BLAND     80           BLAND
370   ELKTONDP      ELKTON        ELKO     80         HENRICO
460   TARBORO5     TARBORO    MARLBORO     80        RICHMOND
489   BRNSWKDP      BRNS

In [25]:
# Look at matches by score band
print("Matches by score band:")
for low, high in [(95,100), (90,95), (85,90), (80,85)]:
    subset = df_matches2[
        (df_matches2["score"] >= low) & 
        (df_matches2["score"] < high) &
        (df_matches2["zone"] == "DOM")
    ]
    print(f"\n--- {low}-{high} ({len(subset)} matches) ---")
    print(subset[["pnode_name", "pnode_clean", "best_match", 
                   "score", "county"]].to_string())

Matches by score band:

--- 95-100 (0 matches) ---
Empty DataFrame
Columns: [pnode_name, pnode_clean, best_match, score, county]
Index: []

--- 90-95 (44 matches) ---
    pnode_name pnode_clean best_match  score          county
4     BANISTE4     BANISTE   BANISTER     93    PITTSYLVANIA
17    CENTRLIA    CENTRLIA  CENTRALIA     94    CHESTERFIELD
33    EMPORADP      EMPORA    EMPORIA     92         EMPORIA
36    FARMVILL    FARMVILL  FARMVILLE     94      CUMBERLAND
55    HOPEWEL4     HOPEWEL   HOPEWELL     93        HOPEWELL
65    KENBRIDG    KENBRIDG  KENBRIDGE     94       LUNENBURG
70    LAKESID4     LAKESID   LAKESIDE     93         HENRICO
100    PEARSON     PEARSON   PEARSONS     93         HANOVER
132   WAKFIELD    WAKFIELD  WAKEFIELD     94     SOUTHAMPTON
159   FENTRES4     FENTRES   FENTRESS     93      CHESAPEAKE
160   FRANKLI4     FRANKLI   FRANKLIN     93        FRANKLIN
165   GREENRUN    GREENRUN  GREEN RUN     94  VIRGINIA BEACH
166   GREENWCH    GREENWCH  GREENWICH   

In [30]:
# Get unmatched DOM pnodes only (AEP unmatched are mostly out of state)
df_unmatched_dom = df_matches2[
    (df_matches2["score"] < 85) &
    (df_matches2["zone"] == "DOM")
].copy()

df_unmatched_aep = df_matches2[
    (df_matches2["score"] < 85) &
    (df_matches2["zone"] == "AEP")
].copy()

print(f"Unmatched DOM pnodes: {len(df_unmatched_dom)}")

# Categorize by name pattern
df_unmatched_dom["pattern"] = "other"
df_unmatched_dom.loc[df_unmatched_dom["pnode_name"].str.endswith("DP"), "pattern"] = "DP suffix"
df_unmatched_dom.loc[df_unmatched_dom["pnode_name"].str.endswith("4"), "pattern"] = "4 suffix"
df_unmatched_dom.loc[df_unmatched_dom["pnode_name"].str.endswith("5"), "pattern"] = "5 suffix"
df_unmatched_dom.loc[df_unmatched_dom["pnode_name"].str.match(r'.*\d$'), "pattern"] = "number suffix"

print("\nPattern breakdown:")
print(df_unmatched_dom["pattern"].value_counts())

print("\nSample DP suffix unmatched:")
print(df_unmatched_dom[df_unmatched_dom["pattern"] == "DP suffix"]["pnode_name"].tolist()[:20])

print("\nSample number suffix unmatched:")
print(df_unmatched_dom[df_unmatched_dom["pattern"] == "number suffix"]["pnode_name"].tolist()[:20])

print("\nSample other unmatched:")
print(df_unmatched_dom[df_unmatched_dom["pattern"] == "other"]["pnode_name"].tolist()[:20])

Unmatched DOM pnodes: 583

Pattern breakdown:
pattern
other            439
DP suffix        107
number suffix     37
Name: count, dtype: int64

Sample DP suffix unmatched:
['BAKRSPDP', 'BARNJNDP', 'BEECHDP', 'BELFLDDP', 'BOYDTNDP', 'BRINKDP', 'CLIMAXDP', 'CRSTLHDP', 'CURDSVDP', 'DANLTNDP', 'GARNERDP', 'GARYDP', 'GASBRGDP', 'GRETNADP', 'GRITDP', 'HICKGRDP', 'HOOPERDP', 'ISLCRKDP', 'KDOMINDP', 'KERRDP']

Sample number suffix unmatched:
['CHESTER4', 'ENON4', 'IRONBRI4', 'MTLAURE4', 'PLESNTH4', 'TURNER4', 'WAVERLY4', 'WAVLYNO2', 'WHITEOA4', 'CRYSTAL4', 'OAKGROV4', 'POSSUMP4', 'STJOHNS4', 'NUCOR4', 'SUNBURY4', 'WOODLAN4', 'EDINBUR4', 'MERCK5', 'MTJACKS4', 'WESTVAC4']

Sample other unmatched:
['BROWNBOV', 'BUCKNGHM', 'CHAPARAL', 'DOMINION', 'FORTLEE', 'FOURRIVR', 'FTPICKET', 'GRETNA', 'HARMONYV', 'HUBER', 'HULLSTR', 'ICI', 'IVOR', 'JETERSVL', 'KINDERTN', 'KINGSLND', 'LANCASTR', 'MARGARTV', 'MAURYST', 'METCALF']


In [27]:
# Export substation names
print("VIRGINIA SUBSTATIONS:")
print(subs_dom_aep["HIFLD_Name"].sort_values().tolist())

print("\nUNMATCHED DOM PNODES:")
print(df_unmatched_dom["pnode_name"].tolist())

VIRGINIA SUBSTATIONS:
['ABINGDON', 'ACCA', 'ALLIED', 'ALPINE', 'ALTAVISTA', 'ALTAVISTA POWER STATION', 'ALUM RIDGE', 'ANNANDALE', 'AQUIA HARBOR', 'ARCOLA', 'ARLINGTON', 'ASHBURN', 'AXTON', 'AXTON', 'BALCONY FALLS', 'BALLSTON', 'BANISTER', 'BARNES JUNCTION', 'BARRACKS ROAD', 'BASIN', 'BATTERY HEIGHTS DP', 'BEAR GARDEN', 'BEARSKIN', 'BEARSKIN', 'BEARWALLOW', 'BEAUMEADE', 'BECO', 'BELLE HAVEN', 'BELLWOOD', 'BELVOIR', 'BENNINGTON', 'BENNS CHURCH', 'BENT MOUNTAIN', 'BIRCHWOOD', 'BIRCHWOOD', 'BLAINE', 'BLAND', 'BLOXOMS CORNER', 'BLUE RIDGE', 'BONSACK', 'BOONSBORO', 'BOXWOOD', 'BOYKINS', 'BRADDOCK', 'BRAMBLETON', 'BREMO BLUFF', 'BRIARFIELD HILLS', 'BROADFORD (138 KV)', 'BROADFORD (765 KV)', 'BRODNAX', 'BROOKVILLE', 'BRUNSWICK', 'BRUSH TAVERN', 'BUCHANAN GENERATION LLC', 'BUCK HYDRO', 'BUCKHORN', 'BULL RUN', 'BURKE-SIDEBURN', 'BURLINGTON', 'BURTON', 'BYLLESBY 2', 'CAMBRIA', 'CANDLERS MOUNTAIN', 'CANNON BRANCH', 'CARSON', 'CARTERSVILLE', 'CARVER', 'CATAWBA', 'CAVE SPRING', 'CELANESE', 'CELANESE

In [28]:
# Get ALL AEP pnodes including matched and unmatched
df_all_aep = df_matches2[df_matches2["zone"] == "AEP"].copy()

print(f"Total AEP pnodes: {len(df_all_aep)}")
print(f"\nScore distribution:")
print(pd.cut(df_all_aep["score"],
             bins=[0, 50, 70, 85, 95, 100])
      .value_counts()
      .sort_index())

# Export all AEP pnode names and their best matches
print("\nALL AEP PNODES AND BEST MATCHES:")
print(df_all_aep[["pnode_name", "pnode_clean", "best_match", 
                   "score", "county", "utility"]]
      .sort_values("score", ascending=False)
      .to_string())

Total AEP pnodes: 1700

Score distribution:
score
(0, 50]       113
(50, 70]     1200
(70, 85]      272
(85, 95]       62
(95, 100]      53
Name: count, dtype: int64

ALL AEP PNODES AND BEST MATCHES:
              pnode_name         pnode_clean          best_match  score          county                       utility
1107              MONETA              MONETA              MONETA    100         BEDFORD          APPALACHIAN POWER CO
872             COLUMBIA            COLUMBIA            COLUMBIA    100        FLUVANNA  VIRGINIA ELECTRIC & POWER CO
2191                LANE                LANE                LANE    100      MONTGOMERY          APPALACHIAN POWER CO
1303            THORNTON            THORNTON            THORNTON    100        FRANKLIN          APPALACHIAN POWER CO
1626             COLLEEN             COLLEEN             COLLEEN    100          NELSON          APPALACHIAN POWER CO
919             EDGEMONT            EDGEMONT            EDGEMONT    100      MONTGOMERY     

In [31]:
# Export substation names
print("VIRGINIA SUBSTATIONS:")
print(subs_dom_aep["HIFLD_Name"].sort_values().tolist())

print("\nUNMATCHED DOM PNODES:")
print(df_unmatched_dom["pnode_name"].tolist())

print("\nUNMATCHED AEP PNODES:")
print(df_unmatched_aep["pnode_name"].tolist())

VIRGINIA SUBSTATIONS:
['ABINGDON', 'ACCA', 'ALLIED', 'ALPINE', 'ALTAVISTA', 'ALTAVISTA POWER STATION', 'ALUM RIDGE', 'ANNANDALE', 'AQUIA HARBOR', 'ARCOLA', 'ARLINGTON', 'ASHBURN', 'AXTON', 'AXTON', 'BALCONY FALLS', 'BALLSTON', 'BANISTER', 'BARNES JUNCTION', 'BARRACKS ROAD', 'BASIN', 'BATTERY HEIGHTS DP', 'BEAR GARDEN', 'BEARSKIN', 'BEARSKIN', 'BEARWALLOW', 'BEAUMEADE', 'BECO', 'BELLE HAVEN', 'BELLWOOD', 'BELVOIR', 'BENNINGTON', 'BENNS CHURCH', 'BENT MOUNTAIN', 'BIRCHWOOD', 'BIRCHWOOD', 'BLAINE', 'BLAND', 'BLOXOMS CORNER', 'BLUE RIDGE', 'BONSACK', 'BOONSBORO', 'BOXWOOD', 'BOYKINS', 'BRADDOCK', 'BRAMBLETON', 'BREMO BLUFF', 'BRIARFIELD HILLS', 'BROADFORD (138 KV)', 'BROADFORD (765 KV)', 'BRODNAX', 'BROOKVILLE', 'BRUNSWICK', 'BRUSH TAVERN', 'BUCHANAN GENERATION LLC', 'BUCK HYDRO', 'BUCKHORN', 'BULL RUN', 'BURKE-SIDEBURN', 'BURLINGTON', 'BURTON', 'BYLLESBY 2', 'CAMBRIA', 'CANDLERS MOUNTAIN', 'CANNON BRANCH', 'CARSON', 'CARTERSVILLE', 'CARVER', 'CATAWBA', 'CAVE SPRING', 'CELANESE', 'CELANESE

In [33]:
import geopandas as gpd

# Load data (already have these from before)
subs = gpd.read_file('/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/electricSubstations/Shapefile/electric_substations_hifld_v4_FieldMods.shp')
territories = gpd.read_file('/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/electricUtilityTerritories/Shapefile/Electric_Retail_Service_Territories_CustomFields.shp')
va_state = gpd.read_file('/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/us_states/tl_2022_us_state.shp')
va_state = va_state[va_state['STUSPS'] == 'VA']

# Align CRS
subs = subs.to_crs('EPSG:4326')
territories = territories.to_crs('EPSG:4326')
va_state = va_state.to_crs('EPSG:4326')

# Get AEP territory polygon(s) for Virginia
# NAME field should contain "APPALACHIAN POWER" or similar
aep_territories = territories[territories['NAME'].str.contains('APPALACHIAN', case=False, na=False)]
print("AEP territory names found:")
print(aep_territories['NAME'].unique())

# Clip AEP territory to Virginia boundary
aep_va = gpd.overlay(aep_territories, va_state[['geometry']], how='intersection')
print(f"\nAEP-VA territory area: {aep_va.geometry.area.sum():.4f} sq degrees")

# Filter substations: within Virginia AND within AEP-VA territory
subs_in_va = gpd.clip(subs, va_state)
subs_aep_va = gpd.clip(subs_in_va, aep_va)

print(f"\nSubstations in Virginia: {len(subs_in_va)}")
print(f"Substations in AEP-Virginia territory: {len(subs_aep_va)}")
print("\nSample AEP-VA substations:")
print(subs_aep_va[['Name', 'HIFLD_Name', 'county', 'max_volt']].head(30).to_string())

AEP territory names found:
<ArrowStringArray>
['APPALACHIAN ELECTRIC COOP', 'APPALACHIAN POWER CO']
Length: 2, dtype: str

AEP-VA territory area: 2.9210 sq degrees

Substations in Virginia: 1432
Substations in AEP-Virginia territory: 365

Sample AEP-VA substations:
                          Name               HIFLD_Name        county  max_volt
5845             Unknown114624            UNKNOWN114624  PITTSYLVANIA -999999.0
32266                Tap146998                TAP146998  PITTSYLVANIA -999999.0
5694             Unknown114451            UNKNOWN114451      DANVILLE -999999.0
49078            Unknown167387            UNKNOWN167387      DANVILLE      69.0
5700             Unknown114457            UNKNOWN114457      DANVILLE -999999.0
32903                Tap147648                TAP147648      DANVILLE -999999.0
5695             Unknown114452            UNKNOWN114452      DANVILLE -999999.0
31688                 Ridgeway                 RIDGEWAY         HENRY     138.0
5691   Danvill

/var/folders/fk/jrs5sj2165v98_klcg800pk40000gn/T/ipykernel_5499/3186425974.py:22: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  print(f"\nAEP-VA territory area: {aep_va.geometry.area.sum():.4f} sq degrees")


In [34]:
# Get DOM territory polygon(s)
dom_territories = territories[territories['NAME'].str.contains('VIRGINIA ELECTRIC', case=False, na=False)]
print("DOM territory names found:")
print(dom_territories['NAME'].unique())

# Clip DOM territory to Virginia boundary
dom_va = gpd.overlay(dom_territories, va_state[['geometry']], how='intersection')

# Filter substations: within Virginia AND within DOM-VA territory
subs_dom_va = gpd.clip(subs_in_va, dom_va)

print(f"\nSubstations in DOM-Virginia territory: {len(subs_dom_va)}")
print("\nSample DOM-VA substations:")
print(subs_dom_va[['Name', 'HIFLD_Name', 'county', 'max_volt']].head(10).to_string())

DOM territory names found:
<ArrowStringArray>
['VIRGINIA ELECTRIC & POWER CO', 'CENTRAL VIRGINIA ELECTRIC COOP']
Length: 2, dtype: str

Substations in DOM-Virginia territory: 814

Sample DOM-VA substations:
                         Name              HIFLD_Name       county  max_volt
32425               Tap147165               TAP147165  MECKLENBURG     115.0
5726            Unknown114494           UNKNOWN114494  MECKLENBURG     115.0
32640                Omega DP                OMEGA DP      HALIFAX     115.0
32649               Tap147393               TAP147393      HALIFAX     115.0
32181                   Welco                   WELCO      HALIFAX     115.0
32180             Reedy Creek             REEDY CREEK      HALIFAX     115.0
46713               Tap161761               TAP161761      HALIFAX     115.0
46712  Halifax County Biomass  HALIFAX COUNTY BIOMASS      HALIFAX     115.0
32179            South Boston            SOUTH BOSTON      HALIFAX     115.0
53446           Unknown

In [35]:
import pandas as pd

# Combine DOM-VA and AEP-VA substations
subs_va_dom_aep = pd.concat([subs_dom_va, subs_aep_va], ignore_index=True)

# Drop duplicates in case any substations fall in overlapping territories
subs_va_dom_aep = subs_va_dom_aep.drop_duplicates(subset='HIFLD_Name')

print(f"DOM-VA substations: {len(subs_dom_va)}")
print(f"AEP-VA substations: {len(subs_aep_va)}")
print(f"Combined (deduplicated): {len(subs_va_dom_aep)}")

# Save as shapefile and CSV
output_dir = '/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/'

# As GeoDataFrame shapefile (preserves geometry)
subs_va_dom_aep.to_file(output_dir + 'va_substations_dom_aep.shp')

# As CSV (drops geometry, keeps attributes)
subs_va_dom_aep.drop(columns='geometry').to_csv(output_dir + 'va_substations_dom_aep.csv', index=False)

print("\nSaved:")
print(f"  {output_dir}va_substations_dom_aep.shp")
print(f"  {output_dir}va_substations_dom_aep.csv")

print("\nColumns available:")
print(subs_va_dom_aep.columns.tolist())

DOM-VA substations: 814
AEP-VA substations: 365
Combined (deduplicated): 1126

Saved:
  /Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/va_substations_dom_aep.shp
  /Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/va_substations_dom_aep.csv

Columns available:
['Name', 'HIFLD_Name', 'HIFLD_ID', 'city', 'City_Prop', 'zip', 'type', 'Type_Prop', 'status', 'Stat_Prop', 'county', 'Count_Prop', 'countyfips', 'StateShort', 'StateFull', 'country', 'latitude', 'longitude', 'naics_code', 'naics_desc', 'source', 'sourcedate', 'val_method', 'val_date', 'lines', 'LineTxt', 'max_volt', 'CurrentMax', 'min_volt', 'CurrentMin', 'max_infer', 'min_infer', 'MinVoltCat', 'geometry']


/Users/elenamurray/miniconda3/envs/mds_thesis/lib/python3.11/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field sourcedate created as String field, though DateTime requested.
  ogr_write(
/Users/elenamurray/miniconda3/envs/mds_thesis/lib/python3.11/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field val_date created as String field, though DateTime requested.
  ogr_write(


In [36]:
# Get all unique AEP pnode names
aep_all_pnodes = df_lmp[df_lmp['zone'] == 'AEP']['pnode_name'].unique().tolist()
print(f"Total unique AEP pnodes: {len(aep_all_pnodes)}")
print(aep_all_pnodes[:20])

Total unique AEP pnodes: 0
[]


In [37]:
print(df_lmp['zone'].unique())
print(df_lmp.columns.tolist())
print(df_lmp.head(3))

<ArrowStringArray>
['DOM']
Length: 1, dtype: str
['datetime_beginning_utc', 'datetime_beginning_ept', 'pnode_id', 'pnode_name', 'voltage', 'equipment', 'type', 'zone', 'system_energy_price_rt', 'total_lmp_rt', 'congestion_price_rt', 'marginal_loss_price_rt', 'row_is_current', 'version_nbr']
  datetime_beginning_utc datetime_beginning_ept  pnode_id pnode_name voltage  \
0    2026-03-19T04:00:00    2026-03-19T00:00:00  34885183       ACCA   13 KV   
1    2026-03-19T04:00:00    2026-03-19T00:00:00  34885185       ACCA   13 KV   
2    2026-03-19T04:00:00    2026-03-19T00:00:00  34885187       ACCA   35 KV   

  equipment  type zone  system_energy_price_rt  total_lmp_rt  \
0       TX1  LOAD  DOM                   58.48         70.97   
1       TX2  LOAD  DOM                   58.48         71.02   
2       TX3  LOAD  DOM                   58.48         71.05   

   congestion_price_rt  marginal_loss_price_rt  row_is_current  version_nbr  
0                10.27                    2.22      

In [40]:
import requests
import os

API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"
url = "https://api.pjm.com/api/v1/rt_hrl_lmps"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

# Get one day of AEP data to extract all pnode names
r = requests.get(url, headers=headers, params={
    "rowCount": 10000,
    "startRow": 1,
    "zone": "AEP",
    "datetime_beginning_ept": "3-19-2026 00:00 to 3-19-2026 23:00",
    "row_is_current": 1,
})

print(r.status_code)
data = r.json()
print(f"Total rows: {data.get('totalRows')}")

df_aep = pd.DataFrame(data.get('items', []))
print(f"Rows returned: {len(df_aep)}")

if len(df_aep) > 0:
    aep_all_pnodes = df_aep['pnode_name'].unique().tolist()
    print(f"Unique AEP pnodes: {len(aep_all_pnodes)}")
    print(sorted(aep_all_pnodes)[:20])

200
Total rows: 70248
Rows returned: 10000
Unique AEP pnodes: 1700
['18THSTHT', '21JOHNST', '21JSTJNC', '21NWHTH', '21SHAWN', '21TREATY', '21WILBRG', '21WWVSTA', '23RDSTRE', '29THST', '32NDSTRE', '3MIM', 'ABERT', 'ABINGDON', 'ACADEMIA', 'ADA', 'ADAMSAEP', 'ADAMSIM', 'ADDISON', 'ADENA']


In [41]:
# Check if totalRows > rows returned
print(f"Total rows available: {data.get('totalRows')}")
print(f"Rows returned: {len(df_aep)}")
print(f"Unique pnodes in first 10k rows: {len(aep_all_pnodes)}")

# Pull remaining pages to check if any new pnode names appear
r2 = requests.get(url, headers=headers, params={
    "rowCount": 10000,
    "startRow": 10001,
    "zone": "AEP",
    "datetime_beginning_ept": "3-19-2026 00:00 to 3-19-2026 23:00",
    "row_is_current": 1,
})

df_aep2 = pd.DataFrame(r2.json().get('items', []))
aep_pnodes_p2 = set(df_aep2['pnode_name'].unique())
aep_pnodes_p1 = set(aep_all_pnodes)

new_pnodes = aep_pnodes_p2 - aep_pnodes_p1
print(f"\nNew pnodes found in page 2: {len(new_pnodes)}")
print(sorted(new_pnodes))

Total rows available: 70248
Rows returned: 10000
Unique pnodes in first 10k rows: 1700

New pnodes found in page 2: 0
[]


In [42]:
from thefuzz import process, fuzz
import re

aep_va_names = subs_aep_va['HIFLD_Name'].dropna().unique().tolist()
print(f"AEP-VA substations: {len(aep_va_names)}")
print(f"AEP pnodes to match: {len(aep_all_pnodes)}")

def clean_pnode(name):
    name = name.strip()
    name = re.sub(r'DP\d*$', '', name).strip()
    name = re.sub(r'AEP$', '', name).strip()
    name = re.sub(r'\d+$', '', name).strip()
    name = re.sub(r'NO\d*$', '', name).strip()
    return name.strip()

results = []
for pnode in aep_all_pnodes:
    cleaned = clean_pnode(pnode)
    match, score = process.extractOne(cleaned, aep_va_names, scorer=fuzz.token_sort_ratio)
    results.append({
        'pnode_name': pnode,
        'cleaned': cleaned,
        'best_match': match,
        'score': score
    })

df_aep_matched = pd.DataFrame(results).sort_values('score', ascending=False)

print("\n=== Score >= 85 ===")
high = df_aep_matched[df_aep_matched['score'] >= 85]
print(f"Count: {len(high)}")
print(high[['pnode_name', 'cleaned', 'best_match', 'score']].to_string())

print("\n=== Score 70-84 ===")
mid = df_aep_matched[(df_aep_matched['score'] >= 70) & (df_aep_matched['score'] < 85)]
print(f"Count: {len(mid)}")
print(mid[['pnode_name', 'cleaned', 'best_match', 'score']].to_string())

print("\n=== Score < 70 ===")
low = df_aep_matched[df_aep_matched['score'] < 70]
print(f"Count: {len(low)}")

AEP-VA substations: 356
AEP pnodes to match: 1700

=== Score >= 85 ===
Count: 88
          pnode_name         cleaned       best_match  score
437         RIDGEWAY        RIDGEWAY         RIDGEWAY    100
191          FOREST2          FOREST           FOREST    100
180         FIELDALE        FIELDALE         FIELDALE    100
536         TAZEWELL        TAZEWELL         TAZEWELL    100
542         THORNTON        THORNTON         THORNTON    100
560           VICKER          VICKER           VICKER    100
562           VINTON          VINTON           VINTON    100
607            WYTHE           WYTHE            WYTHE    100
145          DICKENS         DICKENS          DICKENS    100
622         PHILPOTT        PHILPOTT         PHILPOTT    100
1572          PERMAC          PERMAC           PERMAC    100
627          CLAYTOR         CLAYTOR          CLAYTOR    100
1582          MARVIN          MARVIN           MARVIN    100
642            AXTON           AXTON            AXTON    100
645 

In [43]:
print(subs_aep_va.columns.tolist())
print(subs_aep_va.head(3))

['Name', 'HIFLD_Name', 'HIFLD_ID', 'city', 'City_Prop', 'zip', 'type', 'Type_Prop', 'status', 'Stat_Prop', 'county', 'Count_Prop', 'countyfips', 'StateShort', 'StateFull', 'country', 'latitude', 'longitude', 'naics_code', 'naics_desc', 'source', 'sourcedate', 'val_method', 'val_date', 'lines', 'LineTxt', 'max_volt', 'CurrentMax', 'min_volt', 'CurrentMin', 'max_infer', 'min_infer', 'MinVoltCat', 'geometry']
                Name     HIFLD_Name HIFLD_ID      city City_Prop    zip  \
5845   Unknown114624  UNKNOWN114624   114624   BASSETT   Bassett  24055   
32266      Tap146998      TAP146998   146998      EDEN      Eden  27288   
5694   Unknown114451  UNKNOWN114451   114451  DANVILLE  Danville  24541   

             type   Type_Prop      status   Stat_Prop  ... lines LineTxt  \
5845   SUBSTATION  Substation  IN SERVICE  In service  ...   1.0       1   
32266         TAP         Tap  IN SERVICE  In service  ...   3.0       3   
5694   SUBSTATION  Substation  IN SERVICE  In service  ...   

In [45]:
# Reverse the match — for each of the 365 AEP-VA substations,
# find the best matching pnode from the 1700 AEP pnodes

results_reverse = []
for sub_name in aep_va_names:
    match, score = process.extractOne(sub_name, aep_all_pnodes, scorer=fuzz.token_sort_ratio)
    results_reverse.append({
        'substation': sub_name,
        'best_pnode_match': match,
        'score': score
    })

df_reverse = pd.DataFrame(results_reverse).sort_values('score', ascending=False)

print(f"Total AEP-VA substations: {len(df_reverse)}")
print(df_reverse.to_string())

Total AEP-VA substations: 356
                           substation best_pnode_match  score
96                              FLOYD            FLOYD    100
296                           GOMINGO          GOMINGO    100
107                           CLAYTOR          CLAYTOR    100
264                            GRUNDY           GRUNDY    100
99                              WYTHE            WYTHE    100
92                     JACKSONS FERRY   JACKSONS FERRY    100
90                            LEBANON          LEBANON    100
270                      JOSHUA FALLS     JOSHUA FALLS    100
271                          RUSTBURG         RUSTBURG    100
62                            DICKENS          DICKENS    100
310                          CLIFFORD         CLIFFORD    100
60                            PENHOOK          PENHOOK    100
59                           THORNTON         THORNTON    100
298                             MONEL            MONEL    100
299                           REUSENS   

In [46]:
# Filter out UNKNOWN and TAP substations
aep_va_names_clean = [name for name in aep_va_names 
                      if 'UNKNOWN' not in name.upper() 
                      and 'TAP' not in name.upper()
                      and 'DEADEND' not in name.upper()
                      and 'RISER' not in name.upper()]

print(f"Original AEP-VA substations: {len(aep_va_names)}")
print(f"After filtering: {len(aep_va_names_clean)}")
print(sorted(aep_va_names_clean))

Original AEP-VA substations: 356
After filtering: 156
['ABINGDON', 'ALUM RIDGE', 'AXTON', 'BEARSKIN', 'BEARWALLOW', 'BENNINGTON', 'BENT MOUNTAIN', 'BLAINE', 'BLAND', 'BLUE RIDGE', 'BONSACK', 'BOONSBORO', 'BOXWOOD', 'BROADFORD (138 KV)', 'BROADFORD (765 KV)', 'BROOKVILLE', 'BRUSH TAVERN', 'BUCHANAN GENERATION LLC', 'BUCK HYDRO', 'BUCKHORN', 'BURLINGTON', 'BYLLESBY 2', 'CAMBRIA', 'CANDLERS MOUNTAIN', 'CATAWBA', 'CAVE SPRING', 'CELANESE', 'CELANESE ACETATE LLC', 'CENTERVILLE', 'CLAYPOOL HILL', 'CLAYTOR', 'CLEARBROOK', 'CLIFFORD', 'CLINCH RIVER', 'CLINCHFIELD', 'CLOVERDALE', 'CLOVERDALE 765KV', 'CLOVERDALE EAST', 'COAL CREEK', 'COLLEEN', 'COPPER RIDGE', 'CUSHAW', 'DANVILLE', 'DANVILLE NEW DESIGN PLANT', 'DANVILLE WESTOVER PLANT', 'DEARINGTON', 'DICKENS', 'DISMAL RIVER', 'EDGEMONT', 'ELK GARDEN', 'FALLING BRANCH', 'FIELDALE', 'FLETCHERS RIDGE', 'FLOYD', 'FOREST', 'FRANKLIN', 'FREEMONT', 'FRIES', 'GARDEN CREEK', 'GEORGE STREET', 'GLEN LYN', 'GOMINGO', 'GRAVES MILL', 'GRUNDY', 'HALES BRANCH',

In [47]:
results_reverse = []
for sub_name in aep_va_names_clean:
    match, score = process.extractOne(sub_name, aep_all_pnodes, scorer=fuzz.token_sort_ratio)
    results_reverse.append({
        'substation': sub_name,
        'best_pnode_match': match,
        'score': score
    })

df_reverse = pd.DataFrame(results_reverse).sort_values('score', ascending=False)
print(f"Total named AEP-VA substations: {len(df_reverse)}")
print(df_reverse.to_string())

Total named AEP-VA substations: 156
                           substation best_pnode_match  score
0                            RIDGEWAY         RIDGEWAY    100
136                           BOXWOOD          BOXWOOD    100
33                              WYTHE            WYTHE    100
134                           SKIMMER          SKIMMER    100
133                           REUSENS          REUSENS    100
132                             MONEL            MONEL    100
77                               LANE             LANE    100
39                            CLAYTOR          CLAYTOR    100
40                              WURNO            WURNO    100
46                             BLAINE           BLAINE    100
48                             MONETA           MONETA    100
49                           EDGEMONT         EDGEMONT    100
117                          RUSTBURG         RUSTBURG    100
116                      JOSHUA FALLS     JOSHUA FALLS    100
112                            GRU

In [48]:
# Substations to exclude (wrong or uncertain matches)
exclude_substations = [
    'BLAND', 'NEW LONDON', 'HILL', 'RED HILL', 'LAKE FOREST', 'SANDY RIDGE',
    'SMITH MOUNTAIN', 'OAK LEVEL', 'HURLEY', 'SNOWDEN', 'COAL CREEK',
    'GARDEN CREEK', 'BEARWALLOW', 'PEAK CREEK', 'BUCK HYDRO', 'BEARSKIN',
    'OPOSSUM CREEK', 'FALLING BRANCH', 'PRICES FORK', 'LEE HIGHWAY',
    'CUSHAW', 'WEST SALEM', 'IVY HILL', 'HALES BRANCH', 'JEWELL BRANCH',
    'TURKEY PEN', 'WHITESTONE BRANCH', 'HUNTINGTON COURT',
    'SOUTH CHRISTIANBURG', 'SOUTH CHRISTIANSBURG', 'KNOTTY POPLAR',
    'SKAGGS BRANCH', 'MATT FUNK', 'SHACK MILLS STATION',
    'DANVILLE WESTOVER PLANT', 'SALEM ELECTRIC DEPARTMENT',
    'DANVILLE NEW DESIGN PLANT', 'JEWELL SMOKELESS C AND C COAL CO',
    'RADFORD ARMY AMMUNITION PLANT', 'KEAN MTN AND GRASSY CREEK',
    # Uncertain
    'MOSS', 'CLOVERDALE EAST', 'CLOVERDALE 765KV', 'TANK HILL',
    'DISMAL RIVER', 'RED OAK (AP)', 'KNOX COAL CREEK',
]

# Build confirmed AEP-VA pnode list
df_aep_va_confirmed = df_reverse[
    (df_reverse['score'] >= 70) &
    (~df_reverse['substation'].isin(exclude_substations))
].copy()

print(f"Confirmed AEP-VA pnodes: {len(df_aep_va_confirmed)}")
print(df_aep_va_confirmed[['substation', 'best_pnode_match', 'score']].to_string())

Confirmed AEP-VA pnodes: 99
             substation best_pnode_match  score
0              RIDGEWAY         RIDGEWAY    100
136             BOXWOOD          BOXWOOD    100
33                WYTHE            WYTHE    100
134             SKIMMER          SKIMMER    100
133             REUSENS          REUSENS    100
132               MONEL            MONEL    100
77                 LANE             LANE    100
39              CLAYTOR          CLAYTOR    100
40                WURNO            WURNO    100
46               BLAINE           BLAINE    100
48               MONETA           MONETA    100
49             EDGEMONT         EDGEMONT    100
117            RUSTBURG         RUSTBURG    100
116        JOSHUA FALLS     JOSHUA FALLS    100
112              GRUNDY           GRUNDY    100
55             BUCKHORN         BUCKHORN    100
58             TAZEWELL         TAZEWELL    100
105            CELANESE         CELANESE    100
103             CATAWBA          CATAWBA    100
62          

In [50]:
# Get one day of DOM data to extract all pnode names
r = requests.get(url, headers=headers, params={
    "rowCount": 10000,
    "startRow": 1,
    "zone": "DOM",
    "datetime_beginning_ept": "3-19-2026 00:00 to 3-19-2026 23:00",
    "row_is_current": 1,
})

data_dom = r.json()
print(f"Total rows: {data_dom.get('totalRows')}")

df_dom = pd.DataFrame(data_dom.get('items', []))
dom_all_pnodes = df_dom['pnode_name'].unique().tolist()
print(f"Unique DOM pnodes in first 10k rows: {len(dom_all_pnodes)}")

# Check if page 2 has any new pnodes
r2 = requests.get(url, headers=headers, params={
    "rowCount": 10000,
    "startRow": 10001,
    "zone": "DOM",
    "datetime_beginning_ept": "3-19-2026 00:00 to 3-19-2026 23:00",
    "row_is_current": 1,
})

df_dom2 = pd.DataFrame(r2.json().get('items', []))
new_dom_pnodes = set(df_dom2['pnode_name'].unique()) - set(dom_all_pnodes)
print(f"New pnodes found in page 2: {len(new_dom_pnodes)}")

Total rows: 42120
Unique DOM pnodes in first 10k rows: 761
New pnodes found in page 2: 0


In [52]:
# Get clean DOM-VA substation names
dom_va_names = subs_dom_va['HIFLD_Name'].dropna().unique().tolist()
dom_va_names_clean = [name for name in dom_va_names 
                      if 'UNKNOWN' not in name.upper()
                      and not name.upper().startswith('TAP')
                      and not name.upper().startswith('DEADEND')
                      and not name.upper().startswith('RISER')]

print(f"Original DOM-VA substations: {len(dom_va_names)}")
print(f"After filtering unknowns: {len(dom_va_names_clean)}")

# Reverse match — for each DOM-VA substation find best pnode match
results_dom_reverse = []
for sub_name in dom_va_names_clean:
    match, score = process.extractOne(sub_name, dom_all_pnodes, scorer=fuzz.token_sort_ratio)
    results_dom_reverse.append({
        'substation': sub_name,
        'best_pnode_match': match,
        'score': score
    })

df_dom_reverse = pd.DataFrame(results_dom_reverse).sort_values('score', ascending=False)
print(f"\nTotal named DOM-VA substations: {len(df_dom_reverse)}")
print(df_dom_reverse.to_string())

Original DOM-VA substations: 804
After filtering unknowns: 339

Total named DOM-VA substations: 339
                              substation best_pnode_match  score
127                              OAKWOOD          OAKWOOD    100
202                                TOANO            TOANO    100
73                                VERONA           VERONA    100
72                                 DOOMS            DOOMS    100
194                                HAYES            HAYES    100
70                                CROZET           CROZET    100
251                          MORRISVILLE      MORRISVILLE    100
1                                  WELCO            WELCO    100
275                                   OX               OX    100
244                         POSSUM POINT     POSSUM POINT    100
204                           MIDLOTHIAN       MIDLOTHIAN    100
174                                SEPTA            SEPTA    100
274                             OCCOQUAN         OCCOQU

In [54]:
# All confirmed DOM substation → pnode mappings
confirmed_dom_mappings = {
    # Score 100
    'OAKWOOD': 'OAKWOOD', 'TOANO': 'TOANO', 'VERONA': 'VERONA',
    'DOOMS': 'DOOMS', 'HAYES': 'HAYES', 'CROZET': 'CROZET',
    'MORRISVILLE': 'MORRISVILLE', 'WELCO': 'WELCO', 'OX': 'OX',
    'POSSUM POINT': 'POSSUM POINT', 'MIDLOTHIAN': 'MIDLOTHIAN',
    'SEPTA': 'SEPTA', 'OCCOQUAN': 'OCCOQUAN', 'MAGRUDER': 'MAGRUDER',
    'WARWICK': 'WARWICK', 'REYMET': 'REYMET', 'SHERWOOD': 'SHERWOOD',
    'CUNNINGHAM': 'CUNNINGHAM', 'LANEXA': 'LANEXA', 'BOYKINS': 'BOYKINS',
    'BURTON': 'BURTON', 'TYSONS': 'TYSONS', 'KEVLAR': 'KEVLAR',
    'GROTTOES': 'GROTTOES', 'WALTHALL': 'WALTHALL', 'VALLEY': 'VALLEY',
    'BELVOIR': 'BELVOIR', 'RURITAN': 'RURITAN', 'THALIA': 'THALIA',
    'SIDEBURN': 'SIDEBURN', 'BRADDOCK': 'BRADDOCK', 'PIPELINE': 'PIPELINE',
    'POE': 'POE', 'MYRTLE': 'MYRTLE', 'BELLWOOD': 'BELLWOOD',
    'SUFFOLK': 'SUFFOLK', 'YADKIN': 'YADKIN', 'GLEBE': 'GLEBE',
    'THRASHER': 'THRASHER', 'FENTRESS': 'FENTRESS', 'HICKORY': 'HICKORY',
    'MARLBORO': 'MARLBORO', 'QUANTICO': 'QUANTICO', 'BASIN': 'BASIN',
    'DAYTON': 'DAYTON', 'SHOCKOE': 'SHOCKOE', 'CARVER': 'CARVER',
    'HUNTER': 'HUNTER', 'GOSPORT': 'GOSPORT', 'LEXINGTON': 'LEXINGTON',
    'LOCKS': 'LOCKS', 'ARCOLA': 'ARCOLA', 'SAPONY': 'SAPONY',
    'CARSON': 'CARSON', 'GREENWAY': 'GREENWAY', 'ASHBURN': 'ASHBURN',
    'HCF': 'HCF', 'STAFFORD': 'STAFFORD', 'CHICKAHOMINY': 'CHICKAHOMINY',
    'NIVO': 'NIVO', 'HARVELL': 'HARVELL', 'JARRATT': 'JARRATT',
    'PERTH': 'PERTH', 'CLOVER': 'CLOVER', 'DUMFRIES': 'DUMFRIES',
    'DRYBURG': 'DRYBURG', 'BECO': 'BECO', 'CREWE': 'CREWE',
    'SINAI': 'SINAI', 'TYLER': 'TYLER', 'PLEASANT VIEW': 'PLEASANT VIEW',
    'COLONY': 'COLONY', 'MOSBY': 'MOSBY', 'SEAFORD': 'SEAFORD',
    'EMPORIA': 'EMPORIA', 'PLAZA': 'PLAZA', 'RESTON': 'RESTON',
    'TREGO': 'TREGO', 'LOWMOOR': 'LOWMOOR', 'CIA': 'CIA',
    'SURRY': 'SURRY', 'ACCA': 'ACCA', 'ELMONT': 'ELMONT',
    'ALLIED': 'ALLIED', 'WALNEY': 'WALNEY', 'SULLY': 'SULLY',
    'ALPINE': 'ALPINE', 'CLIFTON': 'CLIFTON', 'DULLES': 'DULLES',
    'WHEELER': 'WHEELER',
    # Score 94 truncated
    'LYNNHAVEN': 'LYNNHAVN', 'WARRENTON': 'WARRENTN',
    'GREENWICH': 'GREENWCH', 'REMINGTON': 'REMINGTN',
    'LAKERIDGE': 'LAKRIDGE', 'CENTRALIA': 'CENTRLIA',
    'LIGHTFOOT': 'LGHTFOOT', 'WAKEFIELD': 'WAKFIELD',
    'FRANCONIA': 'FRANCONA', 'ANNANDALE': 'ANNANDAL',
    'COMORN DP': 'COMORNDP', 'GREEN RUN': 'GREENRUN',
    'CLARENDON': 'CLARENDN', 'COVINGTON': 'COVINGTN',
    'FARMVILLE': 'FARMVILL', 'PENDLETON': 'PENDLETN',
    'ALTAVISTA': 'ALTAVSTA', 'POLAND RD': 'POLANDRD',
    'PENINSULA': 'PENINSLA', 'MINE ROAD': 'MINEROAD',
    'SHELLBANK': 'SHELLBNK', 'KENBRIDGE': 'KENBRIDG',
    'BEAUMEADE': 'BEAUMEAD', 'LANDSTOWN': 'LANDSTWN',
    'OAK RIDGE': 'OAKRIDGE', 'ARLINGTON': 'ARLINGTN',
    'FAIRFIELD': 'FAIRFELD',
    # Score 93-88
    'PEARSONS': 'PEARSON', 'HANOVER': 'HANOVER4',
    'GRAFTON': 'GRAFTON4', 'BALLSTON': 'BALSTON',
    'NEW ROAD': 'NEWROAD', 'HAMILTON': 'HAMILTN',
    'GLASGOW': 'GLASGOW4', 'ROSSLYN': 'ROSLYN',
    'LOUDON': 'LOUDOUN', 'TRABUE': 'TRABUE4',
    'DUPONT': 'DUPONT4', 'GOSHEN': 'GOSHEN4',
    'CLARK': 'CLARK4', 'ROCKBRIDGE': 'ROCKBRDG',
    'CRITTENDEN': 'CRITTNDN', 'KEENE MILL': 'KEENMILL',
    'MANCHESTER': 'MANCHSTR', 'ELKO': 'ELKO4',
    'CHURCHLAND': 'CHRCHLND', 'HARROWGATE': 'HARWGATE',
    'CHASE CITY': 'CHASECTY', 'SMITHFIELD': 'SMITHFLD',
    'BRAMBLETON': 'BRAMBLET', 'DISPUTANTA': 'DISPUTNT',
    'MIDDLEBURG': 'MIDDLEBG', 'WOODBRIDGE': 'WOODBRGE',
    'DUNNSVILLE': 'DUNNSVIL', 'WINTERPOCK': 'WINTERPK',
    'HOPEWELL': 'HOPEWEL4', 'SOUTHWEST': 'STHWEST',
    'YORKTOWN': 'YORKTOW4', 'IDYLWOOD': 'IDYLWOO4',
    'BANISTER': 'BANISTE4', 'LAKESIDE': 'LAKESID4',
    'HOLLYMEAD': 'HOLLYMD', 'COLUMBIA': 'COLUMBI4',
    'HAYFIELD': 'HAYFIEL4', 'MIDWAY': 'MIDWAYDP',
    # Score 84-70 confirmed
    'JETERSVILLE': 'JETERSVL', 'BEAR GARDEN': 'BEARGRDN',
    'CRAIGSVILLE': 'CRAIGSVL', 'DRANESVILLE': 'DRANESVL',
    'TIMBERVILLE': 'TIMBERVL', 'OTTER RIVER': 'OTTERRIV',
    'RAVENSWORTH': 'RAVNSWTH', 'GUM SPRINGS': 'GUMSPRNG',
    'DAVIS DRIVE': 'DAVISDRV', 'GAINESVILLE': 'GAINESVL',
    'CLOVER HILL': 'CLOVERHL', 'FULLER ROAD': 'FULLERRD',
    'MOORE': 'MOOREDP', 'RIVER ROAD': 'RIVERRD',
    'MCLAUGHLIN': 'MCLAUGH', 'FLUVANNA P.S.': 'FLUVANNA',
    'GORDONSVILLE': 'GORDONSV', 'NEWPORT NEWS': 'NWPORTNW',
    'CARTERSVILLE': 'CARTERSV', 'PROFFIT': 'PROFITDP',
    'POLYESTER PS': 'POLYESTR', 'MARSH RUN CT': 'MARSHRUN',
    'FISHERSVILLE': 'FISHERSV', 'HARRISONBURG': 'HARRSNBG',
    'GALLOWS ROAD': 'GALLWSRD', 'SPRUANCE NUG': 'SPRUANCE',
    'REMINGTON CT': 'REMINGTN', 'LOVETTSVILLE': 'LOVETTSV',
    'ELKTON D.P.': 'ELKTONDP', 'BARRACKS ROAD': 'BARRCKRD',
    'DARBYTOWN CTS': 'DARBYTWN', 'LAWRENCEVILLE': 'LAWRNCVL',
    'CLIFTON FORGE': 'CLIFTNFG', 'BENNS CHURCH': 'BENNSCH',
    "KING'S MILL": 'KINGSMIL', 'BLOXOMS CORNER': 'BLOXOMSC',
    'FREDERICKSBURG': 'FRDRKSBG', 'GARNER DP': 'GARNERDP',
    'HARMONY VILLAGE': 'HARMONYV', 'MOUNTAIN ROAD': 'MTNROAD',
    'WESTMORELAND': 'WMORELN4', 'CHARLOTTESVILLE': 'CHARLTSV',
    'CHESTERFIELD': 'CHESTER4', 'INDUSTRIAL PARK': 'INDUSTPK',
    'FORK PICKETT': 'FTPICKET', 'REEVES AVE': 'REEVESAV',
    'TURNER ROAD': 'TURNER4', 'JOHNSON DP': 'JOHNSNDP',
    'WOODS DP': 'WOODSDP', 'OMEGA DP': 'OMEGADP',
    'WINDSOR DP': 'WINDSRDP', 'FREEMAN DP': 'FREEMNDP',
    'NORTH POLE': 'NPOLE', 'BATTERY HEIGHTS DP': 'BATERYHT',
    'CUB-RUN DP': 'CUBRUNDP', 'FALLS CHURCH': 'CHURCHST',
    'PRINCE WILLIAM DP': 'PRINCWIL',
    'CHARLES CITY ROAD': 'CHARLCTY', 'FORT EUSTIS': 'FTEUSTIS',
    'STUMPY LAKE': 'STUMPYLK', 'HULL STREET ROAD': 'HULLSTR',
    'SOUTH BOSTON': 'SOBOSTON', 'THOLE STREET': 'THOLEST',
    'SWINKS MILL': 'SWINKMIL', 'SHORT PUMP': 'SHORTPMP',
    'TWELVTH STREET': 'TWELFTHS', 'RADNOR HEIGHTS': 'RADNORHT',
    'CANNON BRANCH': 'CANNONBR', 'PEACH GROVE D.P.': 'PEACHGDP',
    'NATIONAL WELDERS IND': 'NATWELDR',
    'HARRISON': 'HARRSNDP', 'NORTHEAST': 'NEAST',
    'DANIELTOWN': 'DANLTNDP', 'WHITE OAK': 'WHITEOA4',
    'BREMO BLUFF': 'BREMO4', 'CRANEY ISLAND': 'CRANESCR',
    'SOUTHAMPTON SOLAR, LLC': 'SHAMPTON',
    'NORTH RIVER D.P.': 'NIRIVER',
    'GORDONSVILLE ENERGY LP': 'GORDONSV',
    'CHESAPEAKE ENERGY CENTER': 'CHESAPKE',
    'ELIZABETH RIVER POWER STATION': 'ELIZRIV',
    'MOORE': 'MOOREDP',
    'ALTAVISTA POWER STATION': 'ALTAVSTA',
}

dom_lookup = subs_dom_va.drop_duplicates(subset='HIFLD_Name').set_index('HIFLD_Name')[
    ['latitude', 'longitude', 'county']
].to_dict('index')

dom_rows = []
for substation, pnode in confirmed_dom_mappings.items():
    coords = dom_lookup.get(substation, {})
    dom_rows.append({
        'pnode_name': pnode,
        'substation': substation,
        'latitude': coords.get('latitude'),
        'longitude': coords.get('longitude'),
        'county': coords.get('county'),
        'zone': 'DOM',
    })

df_dom_va_confirmed = pd.DataFrame(dom_rows).drop_duplicates(subset='pnode_name')

print(f"Confirmed DOM-VA pnodes: {len(df_dom_va_confirmed)}")
print(f"With coordinates: {df_dom_va_confirmed['latitude'].notna().sum()}")
print(df_dom_va_confirmed[['pnode_name', 'substation', 'county', 'latitude', 'longitude']].to_string())

Confirmed DOM-VA pnodes: 234
With coordinates: 234
        pnode_name                     substation           county   latitude  longitude
0          OAKWOOD                        OAKWOOD          NORFOLK  36.909789 -76.237335
1            TOANO                          TOANO       JAMES CITY  37.363241 -76.804877
2           VERONA                         VERONA          AUGUSTA  38.189897 -78.985809
3            DOOMS                          DOOMS          AUGUSTA  38.107035 -78.849097
4            HAYES                          HAYES       GLOUCESTER  37.279203 -76.494808
5           CROZET                         CROZET        ALBEMARLE  38.084710 -78.698540
6      MORRISVILLE                    MORRISVILLE         FAUQUIER  38.501088 -77.708789
7            WELCO                          WELCO          HALIFAX  36.671966 -78.920093
8               OX                             OX          FAIRFAX  38.723786 -77.288517
9     POSSUM POINT                   POSSUM POINT   PRINCE 

In [55]:
# Combine DOM and AEP confirmed pnodes
df_va_pnodes_final = pd.concat([df_dom_va_confirmed, df_aep_va_confirmed], ignore_index=True)

# Drop any duplicates on pnode_name
df_va_pnodes_final = df_va_pnodes_final.drop_duplicates(subset='pnode_name')

print(f"DOM confirmed pnodes: {len(df_dom_va_confirmed)}")
print(f"AEP confirmed pnodes: {len(df_aep_va_confirmed)}")
print(f"Combined (deduplicated): {len(df_va_pnodes_final)}")
print(f"With coordinates: {df_va_pnodes_final['latitude'].notna().sum()}")

# Save
output_dir = '/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/'
df_va_pnodes_final.to_csv(output_dir + 'va_pnodes_confirmed.csv', index=False)
print(f"\nSaved to {output_dir}va_pnodes_confirmed.csv")
print(df_va_pnodes_final[['pnode_name', 'substation', 'zone', 'county', 'latitude', 'longitude']].to_string())

DOM confirmed pnodes: 234
AEP confirmed pnodes: 99
Combined (deduplicated): 235
With coordinates: 234

Saved to /Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/va_pnodes_confirmed.csv
        pnode_name                     substation zone           county   latitude  longitude
0          OAKWOOD                        OAKWOOD  DOM          NORFOLK  36.909789 -76.237335
1            TOANO                          TOANO  DOM       JAMES CITY  37.363241 -76.804877
2           VERONA                         VERONA  DOM          AUGUSTA  38.189897 -78.985809
3            DOOMS                          DOOMS  DOM          AUGUSTA  38.107035 -78.849097
4            HAYES                          HAYES  DOM       GLOUCESTER  37.279203 -76.494808
5           CROZET                         CROZET  DOM        ALBEMARLE  38.084710 -78.698540
6      MORRISVILLE                    MORRISVILLE  DOM         FAUQUIER  38.501088 -77.708789
7            WELCO            

In [56]:
# Find the duplicates before dedup
dupes = df_va_pnodes_final[df_va_pnodes_final.duplicated(subset='pnode_name', keep=False)]
print(f"Duplicate pnode entries: {len(dupes)}")
print(dupes[['pnode_name', 'substation', 'zone']].sort_values('pnode_name').to_string())

Duplicate pnode entries: 0
Empty DataFrame
Columns: [pnode_name, substation, zone]
Index: []


In [57]:
print("DOM columns:", df_dom_va_confirmed.columns.tolist())
print("AEP columns:", df_aep_va_confirmed.columns.tolist())

print("\nDOM shape:", df_dom_va_confirmed.shape)
print("AEP shape:", df_aep_va_confirmed.shape)

print("\nAEP sample:")
print(df_aep_va_confirmed.head(5).to_string())

DOM columns: ['pnode_name', 'substation', 'latitude', 'longitude', 'county', 'zone']
AEP columns: ['substation', 'best_pnode_match', 'score']

DOM shape: (234, 6)
AEP shape: (99, 3)

AEP sample:
    substation best_pnode_match  score
0     RIDGEWAY         RIDGEWAY    100
136    BOXWOOD          BOXWOOD    100
33       WYTHE            WYTHE    100
134    SKIMMER          SKIMMER    100
133    REUSENS          REUSENS    100


In [58]:
# Rebuild AEP confirmed with correct structure
aep_lookup = subs_aep_va.drop_duplicates(subset='HIFLD_Name').set_index('HIFLD_Name')[
    ['latitude', 'longitude', 'county']
].to_dict('index')

aep_rows = []
for _, row in df_aep_va_confirmed.iterrows():
    substation = row['substation']
    pnode = row['best_pnode_match']
    coords = aep_lookup.get(substation, {})
    aep_rows.append({
        'pnode_name': pnode,
        'substation': substation,
        'latitude': coords.get('latitude'),
        'longitude': coords.get('longitude'),
        'county': coords.get('county'),
        'zone': 'AEP',
    })

df_aep_va_confirmed_clean = pd.DataFrame(aep_rows).drop_duplicates(subset='pnode_name')

print(f"AEP confirmed pnodes: {len(df_aep_va_confirmed_clean)}")
print(f"With coordinates: {df_aep_va_confirmed_clean['latitude'].notna().sum()}")

# Now combine
df_va_pnodes_final = pd.concat([df_dom_va_confirmed, df_aep_va_confirmed_clean], ignore_index=True)
df_va_pnodes_final = df_va_pnodes_final.drop_duplicates(subset='pnode_name')

print(f"\nDOM: {len(df_dom_va_confirmed)}")
print(f"AEP: {len(df_aep_va_confirmed_clean)}")
print(f"Combined: {len(df_va_pnodes_final)}")
print(f"With coordinates: {df_va_pnodes_final['latitude'].notna().sum()}")

# Save
output_dir = '/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/'
df_va_pnodes_final.to_csv(output_dir + 'va_pnodes_confirmed.csv', index=False)
print(f"\nSaved to {output_dir}va_pnodes_confirmed.csv")

AEP confirmed pnodes: 98
With coordinates: 98

DOM: 234
AEP: 98
Combined: 332
With coordinates: 332

Saved to /Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/va_pnodes_confirmed.csv


In [65]:
import requests
import pandas as pd
import os
import time

API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"
base_url = "https://api.pjm.com/api/v1"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

def fetch_with_retry(params, endpoint="rt_hrl_lmps", max_retries=5):
    full_url = f"{base_url}/{endpoint}"
    for attempt in range(max_retries):
        r = requests.get(full_url, headers=headers, params=params)
        if r.status_code == 200:
            return r.json()
        elif r.status_code == 429:
            wait = 60 * (attempt + 1)
            print(f"\n  Rate limited, waiting {wait}s...")
            time.sleep(wait)
        else:
            print(f"\n  Error {r.status_code}")
            return None
    return None

# Pull just one hour for DOM and AEP to get pnode_ids
all_items = []

for zone in ['DOM', 'AEP']:
    data = fetch_with_retry({
        "rowCount": 10000,
        "startRow": 1,
        "zone": zone,
        "datetime_beginning_ept": "1-1-2025 00:00 to 1-1-2025 00:00",
        "row_is_current": 1,
    })
    
    if data:
        all_items.extend(data.get('items', []))
        print(f"{zone}: {data.get('totalRows')} rows")
    time.sleep(12)

df_lookup = pd.DataFrame(all_items)
print(f"\nTotal rows: {len(df_lookup)}")

# Filter to Virginia pnodes and extract id→name mapping
df_va_lookup = df_lookup[df_lookup['pnode_name'].isin(va_pnode_names)][
    ['pnode_id', 'pnode_name']
].drop_duplicates()

print(f"Virginia pnodes found with IDs: {len(df_va_lookup)}")
print(f"Missing: {set(va_pnode_names) - set(df_va_lookup['pnode_name'])}")

# Merge IDs back into master pnode list
df_va_pnodes_final = df_va_pnodes_final.merge(df_va_lookup, on='pnode_name', how='left')
print(f"Missing pnode_id: {df_va_pnodes_final['pnode_id'].isna().sum()}")
print(df_va_pnodes_final[['pnode_name', 'pnode_id', 'zone', 'county']].head(10).to_string())


DOM: 1647 rows
AEP: 2789 rows

Total rows: 4436
Virginia pnodes found with IDs: 798
Missing: set()
Missing pnode_id: 0
  pnode_name  pnode_id zone      county
0    OAKWOOD  34885949  DOM     NORFOLK
1    OAKWOOD  34885951  DOM     NORFOLK
2    OAKWOOD  34885953  DOM     NORFOLK
3    OAKWOOD  34885955  DOM     NORFOLK
4    OAKWOOD  34885957  DOM     NORFOLK
5      TOANO  34886055  DOM  JAMES CITY
6      TOANO  34886057  DOM  JAMES CITY
7     VERONA  34886895  DOM     AUGUSTA
8     VERONA  34886897  DOM     AUGUSTA
9      DOOMS  35010379  DOM     AUGUSTA


In [66]:
# Look at what differentiates multiple IDs for the same pnode_name
print(df_lookup[df_lookup['pnode_name'] == 'OAKWOOD'][
    ['pnode_id', 'pnode_name', 'voltage', 'equipment', 'type']
].to_string())

     pnode_id pnode_name voltage equipment  type
334  34885949    OAKWOOD   13 KV       TX3  LOAD
335  34885951    OAKWOOD   13 KV       TX4  LOAD
336  34885953    OAKWOOD   35 KV       TX1  LOAD
337  34885955    OAKWOOD   35 KV       TX2  LOAD
338  34885957    OAKWOOD   35 KV       TX5  LOAD


In [67]:
print(df_va_pnodes_final.groupby('pnode_name')['pnode_id'].count().describe())

# How many pnodes have more than one ID?
multi = df_va_pnodes_final.groupby('pnode_name')['pnode_id'].count()
print(f"\nPnodes with 1 ID: {(multi == 1).sum()}")
print(f"Pnodes with 2+ IDs: {(multi > 1).sum()}")
print(f"Max IDs for one pnode: {multi.max()}")
print(f"\nTop 10 pnodes by ID count:")
print(multi.sort_values(ascending=False).head(10))

count    332.000000
mean       2.403614
std        1.964885
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max       25.000000
Name: pnode_id, dtype: float64

Pnodes with 1 ID: 111
Pnodes with 2+ IDs: 221
Max IDs for one pnode: 25

Top 10 pnodes by ID count:
pnode_name
CHESTER4    25
YORKTOW4    12
CLOVER      10
REUSENS      9
HANCOCK      8
MOSBY        7
FIELDALE     7
NEAST        6
DULLES       6
BYLLESBY     6
Name: pnode_id, dtype: int64


In [71]:
API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"
base_url = "https://api.pjm.com/api/v1"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}

# Get all unique pnode IDs for Virginia pnodes
va_pnode_ids = df_va_pnodes_final['pnode_id'].dropna().astype(int).unique().tolist()
id_string = ";".join(str(i) for i in va_pnode_ids)
print(f"Total Virginia pnode IDs: {len(va_pnode_ids)}")

def fetch_one_hour_batched(pnode_ids, date_str, batch_size=50):
    all_items = []
    batches = [pnode_ids[i:i+batch_size] for i in range(0, len(pnode_ids), batch_size)]
    print(f"Total batches: {len(batches)}")
    
    for i, batch in enumerate(batches):
        id_string = ";".join(str(x) for x in batch)
        data = fetch_with_retry({
            "rowCount": 10000,
            "startRow": 1,
            "pnode_id": id_string,
            "datetime_beginning_ept": f"{date_str} 00:00 to {date_str} 00:00",
            "row_is_current": 1,
        })
        
        if data:
            all_items.extend(data.get('items', []))
        
        print(f"  Batch {i+1}/{len(batches)}: {len(data.get('items', []) if data else [])} rows")
        time.sleep(12)
    
    return all_items

# Test with one hour
items = fetch_one_hour_batched(va_pnode_ids, "1-1-2025")
df_test = pd.DataFrame(items)
print(f"\nTotal rows: {len(df_test)}")
print(f"Unique pnode names: {df_test['pnode_name'].nunique()}")
print(f"Unique pnode IDs: {df_test['pnode_id'].nunique()}")

Total Virginia pnode IDs: 798
Total batches: 16
  Batch 1/16: 50 rows
  Batch 2/16: 50 rows
  Batch 3/16: 50 rows
  Batch 4/16: 50 rows
  Batch 5/16: 50 rows
  Batch 6/16: 50 rows
  Batch 7/16: 50 rows
  Batch 8/16: 50 rows
  Batch 9/16: 50 rows
  Batch 10/16: 50 rows
  Batch 11/16: 50 rows
  Batch 12/16: 50 rows
  Batch 13/16: 50 rows
  Batch 14/16: 50 rows
  Batch 15/16: 50 rows
  Batch 16/16: 48 rows

Total rows: 798
Unique pnode names: 332
Unique pnode IDs: 798


In [73]:
df_test.head

<bound method NDFrame.head of     datetime_beginning_utc datetime_beginning_ept    pnode_id pnode_name  \
0      2025-01-01T05:00:00    2025-01-01T00:00:00    34885405      HAYES   
1      2025-01-01T05:00:00    2025-01-01T00:00:00    34885407      HAYES   
2      2025-01-01T05:00:00    2025-01-01T00:00:00    34885459     KEVLAR   
3      2025-01-01T05:00:00    2025-01-01T00:00:00    34885461     KEVLAR   
4      2025-01-01T05:00:00    2025-01-01T00:00:00    34885625     REYMET   
..                     ...                    ...         ...        ...   
793    2025-01-01T05:00:00    2025-01-01T00:00:00  2156113305   MIDWAYAP   
794    2025-01-01T05:00:00    2025-01-01T00:00:00  2156113434   HANSMEAD   
795    2025-01-01T05:00:00    2025-01-01T00:00:00  2156113479   MIDWAYAP   
796    2025-01-01T05:00:00    2025-01-01T00:00:00  2156116592   SCOTTSVI   
797    2025-01-01T05:00:00    2025-01-01T00:00:00  2156116595   SCOTTSVI   

    voltage equipment  type zone  system_energy_price_rt 

In [75]:
import http.client
import urllib.parse
import json

# --- Config ---
API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"
PNODE_NAMES = ["CHICAGO HUB", "AEP-DAYTON HUB", "WESTERN HUB"]  # customize these

START_TIME = "2025-01-01 00:00"
END_TIME   = "2025-01-31 23:00"

for pnode_name in PNODE_NAMES:
    params = urllib.parse.urlencode({
        "rowCount": 10,
        "startRow": 1,
        "sort": "datetime_beginning_ept",
        "order": "Asc",
        "datetime_beginning_ept": f"{START_TIME} to {END_TIME}",
        "pnode_name": pnode_name,
        "subscription-key": API_KEY
    })

    try:
        conn = http.client.HTTPSConnection("api.pjm.com")
        conn.request(
            "GET",
            f"/api/v1/rt_da_monthly_lmps?{params}",
            headers={"Ocp-Apim-Subscription-Key": API_KEY}
        )
        response = conn.getresponse()
        raw = response.read()
        conn.close()

        print(f"\n[{pnode_name}] Status: {response.status} {response.reason}")

        if response.status == 200:
            data = json.loads(raw)
            items = data.get("items", [])
            total_rows = data.get("totalRows", "?")
            print(f"Total rows available: {total_rows} | Showing first {len(items)}\n")
            for i, item in enumerate(items, 1):
                print(f"  Row {i}:")
                for k, v in item.items():
                    print(f"    {k}: {v}")
        else:
            print(f"  Error: {raw.decode('utf-8')[:300]}")

    except Exception as e:
        print(f"  Exception for {pnode_name}: {e}")


[CHICAGO HUB] Status: 200 OK
Total rows available: 744 | Showing first 10

  Row 1:
    datetime_beginning_utc: 2025-01-01T05:00:00
    datetime_beginning_ept: 2025-01-01T00:00:00
    pnode_id: 33092313
    pnode_name: CHICAGO HUB
    voltage: None
    equipment: None
    type: HUB
    zone: None
    system_energy_price_rt: 19.731667
    total_lmp_rt: 18.748442
    congestion_price_rt: 0.003474
    marginal_loss_price_rt: -0.986699
    system_energy_price_da: 21.26
    total_lmp_da: 18.918889
    congestion_price_da: -1.38575
    marginal_loss_price_da: -0.955361
  Row 2:
    datetime_beginning_utc: 2025-01-01T06:00:00
    datetime_beginning_ept: 2025-01-01T01:00:00
    pnode_id: 33092313
    pnode_name: CHICAGO HUB
    voltage: None
    equipment: None
    type: HUB
    zone: None
    system_energy_price_rt: 19.993333
    total_lmp_rt: 19.05564
    congestion_price_rt: 0.031682
    marginal_loss_price_rt: -0.969375
    system_energy_price_da: 20.96
    total_lmp_da: 18.619036
    con

In [76]:
df_va_pnodes_final.head

<bound method NDFrame.head of     pnode_name          substation   latitude  longitude    county zone  \
0      OAKWOOD             OAKWOOD  36.909789 -76.237335   NORFOLK  DOM   
1      OAKWOOD             OAKWOOD  36.909789 -76.237335   NORFOLK  DOM   
2      OAKWOOD             OAKWOOD  36.909789 -76.237335   NORFOLK  DOM   
3      OAKWOOD             OAKWOOD  36.909789 -76.237335   NORFOLK  DOM   
4      OAKWOOD             OAKWOOD  36.909789 -76.237335   NORFOLK  DOM   
..         ...                 ...        ...        ...       ...  ...   
793   PROGRESS       PROGRESS PARK  36.961087 -81.003602     WYTHE  AEP   
794    BASSETT        WEST BASSETT  36.769790 -80.017513     HENRY  AEP   
795   INGERSOL      INGERSOLL RAND  37.344527 -79.920316   ROANOKE  AEP   
796  BROADFORD  BROADFORD (138 KV)  36.920739 -81.682952     SMYTH  AEP   
797   FLETCHER     FLETCHERS RIDGE  37.113978 -82.069053  BUCHANAN  AEP   

       pnode_id  
0      34885949  
1      34885951  
2      34885953

In [79]:
import http.client
import urllib.parse
import json

API_KEY = "60a6ac7a3925418cbdaa7fdfec6d2b90"

params = urllib.parse.urlencode({
    "rowCount": 50000,
    "startRow": 1,
    "zone": "DOM",
    "subscription-key": API_KEY
})

conn = http.client.HTTPSConnection("api.pjm.com")
conn.request(
    "GET",
    f"/api/v1/pnode?{params}",
    headers={"Ocp-Apim-Subscription-Key": API_KEY}
)
response = conn.getresponse()
data = json.loads(response.read())
conn.close()

print(f"Total nodes in DOM zone: {data.get('totalRows')}")

Total nodes in DOM zone: 2318


In [80]:
PNODE_NAMES = [str(name) for name in df_va_pnodes_final['pnode_name'].unique().tolist()]
print(f"Total pnode names: {len(PNODE_NAMES)}")
print(PNODE_NAMES)

Total pnode names: 332
['OAKWOOD', 'TOANO', 'VERONA', 'DOOMS', 'HAYES', 'CROZET', 'MORRISVILLE', 'WELCO', 'OX', 'POSSUM POINT', 'MIDLOTHIAN', 'SEPTA', 'OCCOQUAN', 'MAGRUDER', 'WARWICK', 'REYMET', 'SHERWOOD', 'CUNNINGHAM', 'LANEXA', 'BOYKINS', 'BURTON', 'TYSONS', 'KEVLAR', 'GROTTOES', 'WALTHALL', 'VALLEY', 'BELVOIR', 'RURITAN', 'THALIA', 'SIDEBURN', 'BRADDOCK', 'PIPELINE', 'POE', 'MYRTLE', 'BELLWOOD', 'SUFFOLK', 'YADKIN', 'GLEBE', 'THRASHER', 'FENTRESS', 'HICKORY', 'MARLBORO', 'QUANTICO', 'BASIN', 'DAYTON', 'SHOCKOE', 'CARVER', 'HUNTER', 'GOSPORT', 'LEXINGTON', 'LOCKS', 'ARCOLA', 'SAPONY', 'CARSON', 'GREENWAY', 'ASHBURN', 'HCF', 'STAFFORD', 'CHICKAHOMINY', 'NIVO', 'HARVELL', 'JARRATT', 'PERTH', 'CLOVER', 'DUMFRIES', 'DRYBURG', 'BECO', 'CREWE', 'SINAI', 'TYLER', 'PLEASANT VIEW', 'COLONY', 'MOSBY', 'SEAFORD', 'EMPORIA', 'PLAZA', 'RESTON', 'TREGO', 'LOWMOOR', 'CIA', 'SURRY', 'ACCA', 'ELMONT', 'ALLIED', 'WALNEY', 'SULLY', 'ALPINE', 'CLIFTON', 'DULLES', 'WHEELER', 'LYNNHAVN', 'WARRENTN', 'GR

In [83]:
df_va_pnodes_final.head

<bound method NDFrame.head of     pnode_name          substation   latitude  longitude    county zone  \
0      OAKWOOD             OAKWOOD  36.909789 -76.237335   NORFOLK  DOM   
1      OAKWOOD             OAKWOOD  36.909789 -76.237335   NORFOLK  DOM   
2      OAKWOOD             OAKWOOD  36.909789 -76.237335   NORFOLK  DOM   
3      OAKWOOD             OAKWOOD  36.909789 -76.237335   NORFOLK  DOM   
4      OAKWOOD             OAKWOOD  36.909789 -76.237335   NORFOLK  DOM   
..         ...                 ...        ...        ...       ...  ...   
793   PROGRESS       PROGRESS PARK  36.961087 -81.003602     WYTHE  AEP   
794    BASSETT        WEST BASSETT  36.769790 -80.017513     HENRY  AEP   
795   INGERSOL      INGERSOLL RAND  37.344527 -79.920316   ROANOKE  AEP   
796  BROADFORD  BROADFORD (138 KV)  36.920739 -81.682952     SMYTH  AEP   
797   FLETCHER     FLETCHERS RIDGE  37.113978 -82.069053  BUCHANAN  AEP   

       pnode_id  
0      34885949  
1      34885951  
2      34885953

In [84]:
output_dir = '/Users/elenamurray/Documents/Documents/Repositories/MDS THESIS/Master-Thesis-2026/'
df_va_pnodes_final.to_csv(output_dir + 'va_pnodes_confirmed.csv', index=False)
print(f"Saved {len(df_va_pnodes_final)} rows to va_pnodes_confirmed.csv")
print(df_va_pnodes_final.columns.tolist())
print(df_va_pnodes_final.head())

Saved 798 rows to va_pnodes_confirmed.csv
['pnode_name', 'substation', 'latitude', 'longitude', 'county', 'zone', 'pnode_id']
  pnode_name substation   latitude  longitude   county zone  pnode_id
0    OAKWOOD    OAKWOOD  36.909789 -76.237335  NORFOLK  DOM  34885949
1    OAKWOOD    OAKWOOD  36.909789 -76.237335  NORFOLK  DOM  34885951
2    OAKWOOD    OAKWOOD  36.909789 -76.237335  NORFOLK  DOM  34885953
3    OAKWOOD    OAKWOOD  36.909789 -76.237335  NORFOLK  DOM  34885955
4    OAKWOOD    OAKWOOD  36.909789 -76.237335  NORFOLK  DOM  34885957


In [1]:
df_va_pnodes_final[['pnode_id']].dropna().astype(int).drop_duplicates().to_csv(
    output_dir + 'va_pnode_ids.csv', index=False
)
print(f"Saved {len(df_va_pnodes_final['pnode_id'].dropna().unique())} pnode IDs to va_pnode_ids.csv")

NameError: name 'df_va_pnodes_final' is not defined

In [4]:
import csv
from collections import Counter

# --- Config ---
DOWNLOADED_FILE = "rt_hrl_lmps_2025_apr.csv"
ID_FILE         = "va_pnode_ids.csv"

POSSIBLE_ID_COLUMNS = ["pnode_id", "id", "node_id", "pnodeid", "PNODE_ID", "ID", "NODE_ID"]


# --- Load expected IDs ---
def load_expected_ids(csv_file):
    with open(csv_file, newline="") as f:
        reader = csv.DictReader(f)
        headers = reader.fieldnames
        id_column = None
        for candidate in POSSIBLE_ID_COLUMNS:
            if candidate in headers:
                id_column = candidate
                break
        if not id_column:
            print("Could not detect ID column. Headers:", headers)
            exit()
        f.seek(0)
        reader = csv.DictReader(f)
        ids = {row[id_column].strip() for row in reader if row[id_column].strip()}
    return ids


# --- Load actual IDs from downloaded file ---
def load_actual_data(csv_file):
    id_to_name  = {}   # pnode_id -> pnode_name
    id_counts   = Counter()  # how many rows per pnode_id
    total_rows  = 0

    with open(csv_file, newline="") as f:
        reader = csv.DictReader(f)
        headers = reader.fieldnames
        print(f"Columns in downloaded file: {headers}\n")

        for row in reader:
            pid  = str(row.get("pnode_id", "")).strip()
            name = row.get("pnode_name", "UNKNOWN").strip()
            if pid:
                id_to_name[pid] = name
                id_counts[pid] += 1
                total_rows += 1

    return id_to_name, id_counts, total_rows


# --- Run checks ---
print(f"Loading expected IDs from: {ID_FILE}")
expected_ids = load_expected_ids(ID_FILE)
print(f"Expected IDs: {len(expected_ids)}\n")

print(f"Loading downloaded data from: {DOWNLOADED_FILE}")
id_to_name, id_counts, total_rows = load_actual_data(DOWNLOADED_FILE)
actual_ids = set(id_counts.keys())

print(f"Total rows in file:       {total_rows:,}")
print(f"Unique pnode IDs found:   {len(actual_ids)}")

# Core checks
missing  = expected_ids - actual_ids
extra    = actual_ids - expected_ids

print(f"\n{'='*60}")
print(f"RESULTS")
print(f"{'='*60}")
print(f"Expected IDs:      {len(expected_ids)}")
print(f"Found in download: {len(actual_ids)}")
print(f"Missing:           {len(missing)}")
print(f"Unexpected extras: {len(extra)}")

# Row count check — April 2025 has 720 hours (30 days x 24)
expected_hours = 720
print(f"\n{'='*60}")
print(f"ROW COUNT PER NODE (expected {expected_hours} rows each)")
print(f"{'='*60}")

wrong_count = {pid: count for pid, count in id_counts.items()
               if count != expected_hours}

if not wrong_count:
    print(f"✓ All {len(actual_ids)} nodes have exactly {expected_hours} rows.")
else:
    print(f"✗ {len(wrong_count)} nodes have unexpected row counts:")
    print(f"  {'pnode_id':<15} {'pnode_name':<30} {'rows':>6} {'expected':>8} {'diff':>6}")
    print(f"  {'-'*65}")
    for pid, count in sorted(wrong_count.items(), key=lambda x: x[1]):
        name = id_to_name.get(pid, "UNKNOWN")
        diff = count - expected_hours
        print(f"  {pid:<15} {name:<30} {count:>6} {expected_hours:>8} {diff:>+6}")

# Missing IDs
if missing:
    print(f"\n{'='*60}")
    print(f"MISSING IDs ({len(missing)})")
    print(f"{'='*60}")
    for pid in sorted(missing):
        print(f"  {pid}")
else:
    print(f"\n✓ All {len(expected_ids)} expected IDs are present in the download.")

# Unexpected extras
if extra:
    print(f"\n{'='*60}")
    print(f"UNEXPECTED EXTRA IDs ({len(extra)})")
    print(f"{'='*60}")
    for pid in sorted(extra):
        name = id_to_name.get(pid, "UNKNOWN")
        print(f"  {pid:<15} {name}")
else:
    print(f"✓ No unexpected extra IDs found.")

# Final verdict
print(f"\n{'='*60}")
print(f"VERDICT")
print(f"{'='*60}")
if not missing and not extra and not wrong_count:
    print(f"✓ All checks passed — download looks correct.")
    print(f"  {len(actual_ids)} nodes x {expected_hours} hours = {total_rows:,} rows")
else:
    print(f"✗ Some checks failed — review issues above before proceeding.")

Loading expected IDs from: va_pnode_ids.csv
Expected IDs: 798

Loading downloaded data from: rt_hrl_lmps_2025_apr.csv
Columns in downloaded file: ['datetime_beginning_utc', 'datetime_beginning_ept', 'pnode_id', 'pnode_name', 'voltage', 'equipment', 'type', 'zone', 'system_energy_price_rt', 'total_lmp_rt', 'congestion_price_rt', 'marginal_loss_price_rt', 'row_is_current', 'version_nbr']

Total rows in file:       574,560
Unique pnode IDs found:   798

RESULTS
Expected IDs:      798
Found in download: 798
Missing:           0
Unexpected extras: 0

ROW COUNT PER NODE (expected 720 rows each)
✓ All 798 nodes have exactly 720 rows.

✓ All 798 expected IDs are present in the download.
✓ No unexpected extra IDs found.

VERDICT
✓ All checks passed — download looks correct.
  798 nodes x 720 hours = 574,560 rows
